# **A Complex Network Analysis of the European Air Transportation System**

**Authors:** Cindy Chen, Maria del Pilar Fernandez Barrera, Keith Megara Osores Vicuña 
  
**Master:** Health Data Science

**Subject:** Complex Networks

**Course:** 2025-2026

---

## **Table of Contents**

**1. [Data description and pre-processing](#section1)** 

**2. [Network construction](#section2)**   

**3. [Network characterization](#section3)**   
&emsp; **3.1 [Macroscopic characterization](#section31)**  
&emsp; **3.2 [Microscopic characterization](#section32)**  
&emsp; **3.3 [Community detection](#section33)**  

**4. [Network sparsification](#section4)**   
&emsp; **4.1 [Weights thresholding](#section41)**  
&emsp; **4.2 [Disparity filter](#section42)**  

**5. [Robustness and percolation analysis](#section5)**  
&emsp; **5.1 [Random failure](#section51)**   
&emsp; **5.2 [Targeted attack (degree)](#section52)**   
&emsp; **5.3 [Targeted attack (betweenness)](#section53)**   
&emsp; **5.4 [Results of percolation analysis](#section54)**   

**6. [Difussion dynamics](#section6)**  
&emsp; **6.1 [Single random walk](#section61)**   
&emsp; **6.2 [Multiple random walks](#section62)**  
&emsp; **6.3 [Time dependent occupation probability](#section63)**  
&emsp; **6.4 [First passage time](#section64)**  
&emsp; **6.5 [Failure impact analysis](#section65)**  

**7. [Optimal routing and flight strategies](#section7)**   

---

**Important note:** This notebook was run using Windows operation system, conda 4.12.0 version and Python 3.11.14 version (compatible with infomap). In this case, please, notice that to execute this code properly, an Ubuntu teminal must be installed. Then, a virtual environment "a3_venv" (in our case) must be created to install the dependencies (packages) on conda included in the text file titled "requirements.txt", which can be achieved by running the following command: ```pip install -r /path/to/requirements.txt``` (changing /path/to/ to your path to .txt file).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import networkx as nx
# import igraph as ig
# import community as community_louvain
# import leidenalg as la
# from infomap import Infomap

import random
import ast
import statistics
from scipy.stats import linregress
from scipy.sparse.linalg import eigsh
from sklearn.cluster import SpectralClustering

import os
import warnings
from collections import Counter, defaultdict

import folium
from folium import plugins
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage

import imageio.v2
from PIL import Image as PILImage

warnings.filterwarnings("ignore")

<a id='section1'></a>
## **1. Data description and pre-processing**   

For this project, the [European Airports & Air Routes](https://www.kaggle.com/datasets/lunthu/european-airlines-routes) dataset from Kaggle was used. This dataset contains information on airports and possible routes across 260 major airports in Europe.

In [ ]:
# Load data
df = pd.read_csv("./data/europe_air_routes.csv")
display(df)

In [ ]:
# Print column names
print(df.columns)

The dataset consists of 33 columns, where each row represents a direct air route between two European airports.

- **id:** unique identifier for each route in the dataset.
- **iata_from:** International Air Transport Association (IATA) code of the departure airport.
- **iata_to:** IATA code of the arrival airport.
- **price:** estimated ticket price for the route (in US dollars).  
- **day1:** operational status of the route on Monday.  
    - yes: the route is available on this day.
    - no: the route is not operated on this day.
    - upcoming: the route is planned to operate on that day.
- **day2:** indicates whether flights operate on Tuesday.  
- **day3:** indicates whether flights operate on Wednesday.  
- **day4:** indicates whether flights operate on Thursday.  
- **day5:** indicates whether flights operate on Friday.  
- **day6:** indicates whether flights operate on Saturday.  
- **day7:** indicates whether flights operate on Sunday.  
- **flights_per_day:** range of flights operating per day on this route 
- **common_duration:** approximate flight duration for this route (in minutes).  
- **flights_per_week:** number of flights operating per week on this route  
- **first_flight:** date of the first recorded flight for this route (in MM/DD/YY)   
- **last_flight:** date of the last recorded flight for this route (in MM/DD/YY)   
- **airlineroutes:** detailed information about airline operators serving the route. Stored in a list of dictionaries. 
- **arrival_airport_country_code:** country code of the arrival airport. 
- **arrival_airport_name:** name of the arrival airport. 
- **arrival_airport_city_name:** name of the city where the arrival airport is located.  
- **arrival_airport_city_name_en:** English name of the city where the arrival airport is located.  
- **arrival_airport_country:** country of the arrival airport.   
- **arrival_airport_no_routes:** number of distinct routes connected to the arrival airport.
- **departure_city:** name of the city where the departure airport is located.  
- **departure_country:** name of the country where the departure airport is located.  
- **departure_ICAO:** International Civil Aviation Organization (ICAO) code of the departure airport. 
- **departure_latitude:** latitude coordinate of the departure airport. 
- **departure_longitude:** longitude coordinate of the departure airport. 
- **departure_altitude:** altitude coordinate of the departure airport (in feet). 
- **arrival_ICAO:** ICAO code of the arrival airport.  
- **arrival_latitude:** latitude coordinate of the arrival airport. 
- **arrival_longitude:** longitude coordinate of the arrival airport.   
- **arrival_altitude:** altitude coordinate of the arrival airport (in feet).  

In [ ]:
# Filter df to only keep rows with both departure and arrival airports in Europe
europe_countries = [
    "Albania",
    "Andorra",
    "Armenia",
    "Austria",
    "Azerbaijan",
    "Belarus",
    "Belgium",
    "Bosnia and Herzegovina",
    "Bulgaria",
    "Croatia",
    "Cyprus",
    "Czech Republic",
    "Denmark",
    "Estonia",
    "Finland",
    "France",
    "Georgia",
    "Germany",
    "Greece",
    "Hungary",
    "Iceland",
    "Ireland",
    "Italy",
    "Kosovo",
    "Latvia",
    "Liechtenstein",
    "Lithuania",
    "Luxembourg",
    "Macedonia",
    "Macedonia (FYROM)",
    "Malta",
    "Moldova",
    "Monaco",
    "Montenegro",
    "Netherlands",
    "North Macedonia",
    "Norway",
    "Poland",
    "Portugal",
    "Republic of Moldova",
    "Republic of Serbia",
    "Romania",
    "San Marino",
    "Serbia",
    "Slovakia",
    "Slovenia",
    "Spain",
    "Sweden",
    "Switzerland",
    "Ukraine",
    "United Kingdom",
    "Vatican City",
]

df = df[
    df["departure_country"].isin(europe_countries)
    & df["arrival_airport_country"].isin(europe_countries)
]

print(df.shape)

In [ ]:
# Add a departure_airport_name column
df = df.copy()

# Create mapping from IATA to airport name using arrival airports
iata_to_name = (
    df[["iata_to", "arrival_airport_name"]]
    .drop_duplicates()
    .set_index("iata_to")["arrival_airport_name"]
    .to_dict()
)

# Add departure_airport_name column
df.loc[:, "departure_airport_name"] = df["iata_from"].map(iata_to_name)

In [ ]:
# Extract airlines info for each row
airlines_info_list = []

for routes_str in df["airlineroutes"]:
    airlines_dict = {}

    # Convert to list
    for route in ast.literal_eval(routes_str):
        airlines_name = route.get("carrier_name")

        if isinstance(airlines_name, str):
            airline_info = route.get("airline", {})

            # Create the carrier dictionary
            airlines_dict[airlines_name] = {
                "id": airline_info.get("id"),
                "ICAO": airline_info.get("ICAO"),
                "name": airline_info.get("name"),
                "country": airline_info.get("country"),
                "flights_last_24_hours": airline_info.get("flights_last_24_hours"),
                "is_lowcost": airline_info.get("is_lowcost"),
            }

    airlines_info_list.append(airlines_dict)

# Add to df
df["airlines_info"] = airlines_info_list

# Combine all rows into a single dictionary
combined_airlines_info = {}
for row_dict in df["airlines_info"]:
    combined_airlines_info.update(row_dict)

# Extract airline names for each row
airline_names_list = []

for row_dict in df["airlines_info"]:
    airline_names = ", ".join(row_dict.keys())
    airline_names_list.append(airline_names)

df["airline_names"] = airline_names_list

# Get unique airlines
all_airline_names = df["airline_names"].str.split(",").explode().str.strip()
unique_airlines = all_airline_names.unique()

# Sort airline names
sorted_airline_names = sorted([name for name in unique_airlines if name])

print(f"""Initially, there were 260 airlines in the dataset.
\nHowever, after filtering only those routes with inflights and outflights from European airports,
the number of airlines were reduced {len(sorted_airline_names)}.""")

display(
    df[
        [
            "departure_country",
            "arrival_airport_country",
            "airline_names",
            "airlines_info",
        ]
    ].tail(10)
)

We examine the airlines operating for all the different routes and plot the 15 most repeated airlines.

In [ ]:
# Clean and expand the airline data
# We split the 'airline_names' column by comma and explode it so each airline
# gets its own row, effectively counting every route they participate in
df_expanded = df["airline_names"].str.split(", ").explode()

# Count the occurrences (routes) per airline
airline_counts = df_expanded.value_counts().head(15)

plt.figure(figsize=(8, 6))
airline_counts.plot(kind="bar", color="#377eb8", edgecolor="white")

plt.title("Top 15 Airlines by Number of Routes", fontsize=12, fontweight="bold")
plt.xlabel("Airline name", fontsize=10)
plt.ylabel("Number of routes", fontsize=10)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Track unique airlines using a dictionary {airline_id: is_lowcost}
unique_airlines = {}

for entry in df["airlines_info"]:
    # Convert string representation of dict to actual dict
    if isinstance(entry, str):
        try:
            airline_dict = ast.literal_eval(entry)
        except:
            continue
    else:
        airline_dict = entry

    for airline_name, details in airline_dict.items():
        airline_id = details.get("id")
        is_lowcost = details.get("is_lowcost")

        # Store the airline status using its ID as the unique key
        if airline_id not in unique_airlines:
            unique_airlines[airline_id] = is_lowcost

# Count the unique statuses
status_values = list(unique_airlines.values())
low_cost_count = status_values.count(1)
traditional_count = status_values.count(0)

# Visualization
labels = ["Low cost", "Traditional"]
counts = [low_cost_count, traditional_count]

plt.figure(figsize=(8, 6))
bars = plt.bar(
    labels, counts, color=["#ff7f00", "#377eb8"], edgecolor="white", alpha=0.8
)

plt.title("Total Number of Unique Airlines by Type", fontsize=12, fontweight="bold")
plt.ylabel("Number of unique airlines", fontsize=10)

# Add numeric labels on top of the bars
for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        yval + 0.5,
        int(yval),
        ha="center",
        va="bottom",
    )

plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Total unique low cost airlines: {low_cost_count}")
print(f"Total unique traditional airlines: {traditional_count}")

In [ ]:
# Select relevant columns
cols_to_keep = [
    "iata_from",
    "departure_airport_name",
    "departure_city",
    "departure_country",
    "departure_latitude",
    "departure_longitude",
    "departure_altitude",
    "iata_to",
    "arrival_airport_name",
    "arrival_airport_city_name_en",
    "arrival_airport_country",
    "arrival_latitude",
    "arrival_longitude",
    "arrival_altitude",
    "flights_per_week",
    "common_duration",
    "price",
    "airline_names",
]

df = df[cols_to_keep]

In [ ]:
# Rename columns
rename_cols = {
    "iata_from": "departure_iata",
    "iata_to": "arrival_iata",
    "arrival_airport_city_name_en": "arrival_city",
    "arrival_airport_country": "arrival_country",
    "arrival_airport_no_routes": "arrival_num_routes",
}

df = df.rename(columns=rename_cols)

In [ ]:
# Dataframe shape
print(df.shape)

In [ ]:
# Check missing values
df.isna().sum()

In [ ]:
# Check airports with missing data
print(df[df["arrival_latitude"].isna()]["arrival_iata"].unique())

Missing data extracted from the folowing webpage: https://www.airfleets.net/airport/airport-list.htm
1. **GHV:** Brașov-Ghimbav International Airport in Romania.
2. **KVO:** Morava Airport in Serbia.
3. **SCR:** Scandinavian Mountains Airport in Sweden.
4. **VLO:** Vlora International Airport in Albania.
5. **FDE:** Førde Airport (Enkelvåg Airport) in Norway.
6. **JCU:** Ceuta Heliport in Ceuta, Spain.

In [ ]:
# Dictionary with missing airport info
missing_data = {
    "GHV": {
        "arrival_latitude": 45.6945,
        "arrival_longitude": 25.5178,
        "arrival_altitude": 1740,
    },
    "KVO": {
        "arrival_latitude": 43.8186,
        "arrival_longitude": 20.5852,
        "arrival_altitude": 686,
    },
    "SCR": {
        "arrival_latitude": 61.1561,
        "arrival_longitude": 12.8374,
        "arrival_altitude": 1608,
    },
    "VLO": {
        "arrival_latitude": 40.6055,
        "arrival_longitude": 19.4261,
        "arrival_altitude": 8,
    },
    "FDE": {
        "arrival_latitude": 61.3911,
        "arrival_longitude": 5.7569,
        "arrival_altitude": 1045,
    },
    "JCU": {
        "arrival_latitude": 35.8925,
        "arrival_longitude": -5.3058,
        "arrival_altitude": 6.6,
    },
}

# Fill missing values
for iata, info in missing_data.items():
    df.loc[
        df["arrival_iata"] == iata,
        ["arrival_latitude", "arrival_longitude", "arrival_altitude"],
    ] = (info["arrival_latitude"], info["arrival_longitude"], info["arrival_altitude"])

In [ ]:
# Check variable types
df.info()

We verify if the routes from the dataset are unique:

In [ ]:
# Group by both departure and arrival points
route_counts = (
    df.groupby(["departure_iata", "arrival_iata"])
    .size()
    .reset_index(name="route_count")
)

# Filter routes with duplicates (route_count > 1)
duplicates = route_counts[route_counts["route_count"] > 1]

# Display duplicate routes
print("Duplicate routes:")
print(duplicates)

**Data inspection**

In [ ]:
print(f"Total routes: {len(df)}")
print(f"\nUnique departure airports: {df['departure_iata'].nunique()}.")
print(f"\nUnique departure countries: {df['departure_country'].nunique()}.")
print(f"\nUnique arrival airports: {df['arrival_iata'].nunique()}.")
print(f"\nUnique arrival countries: {df['arrival_country'].nunique()}.")
print(f"\nAverage price: ${df['price'].mean():.2f}.")
print(f"\nAverage duration: {df['common_duration'].mean():.0f} minutes.")
print(f"\nAverage flights per week: {df['flights_per_week'].mean():.1f}.")

In [ ]:
print(f"Summary statistics:")
display(df[["price", "common_duration", "flights_per_week"]].describe())

The columns price and flights_per_week having a minimum value of 0 is not accurate. Either the data was missing or those flights are non-existent. Then, we proceed to delete them.

In [ ]:
df_null_flights_w = df[df["flights_per_week"] == 0]
df_null_price = df[df["price"] == 0]

print(f"Number of flights with 0 flights per week: {df_null_flights_w.shape[0]}.")
print(f"Number of flights with a price of $0: {df_null_price.shape[0]}.")

In [ ]:
departure_null_w = df_null_flights_w["departure_iata"].unique().tolist()
arrival_null_w = df_null_flights_w["arrival_iata"].unique().tolist()

print(
    f"Departure airports with zero flights per week ({len(departure_null_w)}). First 10:\n  {departure_null_w[0:10]}."
)
print(
    f"Arrival airports with zero flights per week ({len(arrival_null_w)}). First 10:\n  {arrival_null_w[0:10]}.\n"
)

departure_null_price = df_null_price["departure_iata"].unique().tolist()
arrival_null_price = df_null_price["arrival_iata"].unique().tolist()

print(
    f"Departure airports with flights price as $0 ({len(departure_null_price)}). First 10:\n  {departure_null_price[0:10]}."
)
print(
    f"Arrival airports with flights price as $0 ({len(arrival_null_price)}). First 10:\n  {arrival_null_price[0:10]}."
)

In [ ]:
print(f"Previous size: {df.shape}.")

# We drop those rows with 0 values in the three previously analysed columns
df = df[~((df["flights_per_week"] == 0) | (df["price"] == 0))]

print(f"Size after data cleaning: {df.shape}.")

In [ ]:
print(f"Updated summary statistics:")
display(df[["price", "common_duration", "flights_per_week"]].describe())

In [ ]:
print("CLEANED DATASET:")
print(f"\nTotal routes: {len(df)}")
print(f"\nUnique departure airports: {df['departure_iata'].nunique()}.")
print(f"\nUnique departure countries: {df['departure_country'].nunique()}.")
print(f"\nUnique arrival airports: {df['arrival_iata'].nunique()}.")
print(f"\nUnique arrival countries: {df['arrival_country'].nunique()}.")
print(f"\nAverage price: ${df['price'].mean():.2f}.")
print(f"\nAverage duration: {df['common_duration'].mean():.0f} minutes.")
print(f"\nAverage flights per week: {df['flights_per_week'].mean():.1f}.")

In [ ]:
# Top 10 busiest routes by flights per week
top_routes = df.nlargest(10, "flights_per_week")[
    ["departure_city", "arrival_city", "flights_per_week", "price"]
]

top_routes = top_routes.reset_index(drop=True)
print(f"Top 10 busiest routes by flights per week:")
display(top_routes)

In [ ]:
# Top 10 airports with the most departure routes
most_connected_departures = (
    df["departure_airport_name"].value_counts().head(10).reset_index()
)
most_connected_departures.columns = ["departure_airport_name", "num_routes"]

most_connected_departures = pd.DataFrame(most_connected_departures)
print("Top 10 airports with the most departure routes:")
display(most_connected_departures)

In [ ]:
# Top 10 airports with the most arrival routes
most_connected_arrivals = (
    df["arrival_airport_name"].value_counts().head(10).reset_index()
)
most_connected_arrivals.columns = ["arrival_airport_name", "num_routes"]

most_connected_arrivals = pd.DataFrame(most_connected_arrivals)
print("Top 10 airports with the most arrival routes:")
display(most_connected_arrivals)

In [ ]:
# Top 10 average price by departure country
avg_price_by_country = (
    df.groupby("departure_country")["price"]
    .mean()
    .round(2)
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

avg_price_by_country.columns = ["departure_country", "avg_price"]

# Display the result as a DataFrame
avg_price_by_country = pd.DataFrame(avg_price_by_country)
print("Top 10 average price ($) by departure country:")
display(avg_price_by_country)

In [ ]:
# Final dataframe
display(df.head(5))

In [ ]:
df.to_csv("./data/cleaned_df.csv", index=False)

**Interactive data visualization**

For the visualization, we used as a base the code from here: https://www.kaggle.com/code/lunthu/europe-flight-routes-map. In order to interact with the widget, the code must be run at least until this section.

In [ ]:
# Limit to random 4000 routes for better performance
if len(df) > 4000:
    df_sample = df.sample(n=4000, random_state=42)
    print(f"Displaying random sample of 4000 routes out of {len(df)} total routes.")
else:
    df_sample = df
    print(f"Displaying all {len(df)} routes.")

In [ ]:
# Create labels
departure_label = widgets.HTML(value="<b>Departure:</b>")
arrival_label = widgets.HTML(value="<b>Arrival:</b>")

# Create dropdowns
departure_dropdown = widgets.Dropdown(
    options=["All"] + sorted(df["departure_iata"].unique())
)

arrival_dropdown = widgets.Dropdown(
    options=["All"] + sorted(df["arrival_iata"].unique())
)

# Create button
update_button = widgets.Button(description="Show Map")
output = widgets.Output()


def update_map(b):
    # Clear the previous output before displaying a new map
    with output:
        clear_output(wait=True)

        # Filter data based on dropdown selections
        df_filtered = df_sample.copy()

        if departure_dropdown.value != "All":
            df_filtered = df_filtered[
                df_filtered["departure_iata"] == departure_dropdown.value
            ]

        if arrival_dropdown.value != "All":
            df_filtered = df_filtered[
                df_filtered["arrival_iata"] == arrival_dropdown.value
            ]

        print(f"Showing {len(df_filtered)} routes")

        # Create a new map with location in the centre of Europe
        europe_map = folium.Map(
            location=[50.0, 10.0], zoom_start=4, tiles="CartoDB positron"
        )

        # Add route lines
        for idx, row in df_filtered.iterrows():
            departure_coords = [row["departure_latitude"], row["departure_longitude"]]
            arrival_coords = [row["arrival_latitude"], row["arrival_longitude"]]
            popup_text = f"""
            <b>Route:</b> {row["departure_iata"]} → {row["arrival_iata"]}<br>
            <b>From:</b> {row.get("departure_city", "N/A")}, {row.get("departure_country", "N/A")}<br>
            <b>To:</b> {row.get("arrival_city", "N/A")}, {row.get("arrival_country", "N/A")}<br>
            <b>Price:</b> ${row["price"]}<br>
            <b>Duration:</b> {row["common_duration"]} min<br>
            <b>Flights/week:</b> {row["flights_per_week"]}
            """

            folium.PolyLine(
                locations=[departure_coords, arrival_coords],
                color="blue",
                weight=2.5,
                opacity=0.4,
                popup=folium.Popup(popup_text, max_width=300),
            ).add_to(europe_map)

        # Add arrival airport markers
        arrival_airports = (
            df_filtered.groupby(
                [
                    "arrival_iata",
                    "arrival_latitude",
                    "arrival_longitude",
                    "arrival_city",
                    "arrival_country",
                ]
            )
            .size()
            .reset_index(name="num_routes")
        )

        for idx, airport in arrival_airports.iterrows():
            folium.CircleMarker(
                location=[airport["arrival_latitude"], airport["arrival_longitude"]],
                radius=5,
                popup=f"""
                <b>{airport["arrival_iata"]}</b><br>
                {airport["arrival_city"]}, {airport["arrival_country"]}<br>
                """,
                color="green",
                fill=True,
                fillColor="green",
                fillOpacity=0.6,
            ).add_to(europe_map)

        # Add layer control and fullscreen toggle
        folium.LayerControl().add_to(europe_map)
        plugins.Fullscreen().add_to(europe_map)

        # Display the updated map
        display(europe_map)


# Remove any existing callback
update_button.on_click(update_map)

# Arrange widgets with custom labels
ui = widgets.VBox(
    [
        departure_label,
        departure_dropdown,
        arrival_label,
        arrival_dropdown,
        update_button,
    ]
)
display(ui, output)

---
<a id='section2'></a>
## **2. Network construction**  

To model the European air transportation system as a complex network, we represent the dataset as a directed graph in which nodes correspond to airports and edges represent direct air routes between them. The network is built using data from 260 major European airports, with each directed edge indicating a flight connection from a departure airport to an arrival airport. Node attributes include geographical and contextual information such as location, city, and country, while edges are weighted with operational characteristics including the number of flights per week, average price, and typical flight duration. 

In [ ]:
# Create directed graph
G = nx.DiGraph()

airports = {}

for _, row in df.iterrows():
    # Departure airport
    dep_iata = row["departure_iata"]

    if dep_iata not in airports:
        airports[dep_iata] = {
            "name": row["departure_airport_name"],
            "city": row["departure_city"],
            "country": row["departure_country"],
            "latitude": float(row["departure_latitude"]),
            "longitude": float(row["departure_longitude"]),
            "altitude": float(row["departure_altitude"]),
        }

    # Arrival airport
    arr_iata = row["arrival_iata"]
    if arr_iata not in airports:
        airports[arr_iata] = {
            "name": row["arrival_airport_name"],
            "city": row["arrival_city"],
            "country": row["arrival_country"],
            "latitude": float(row["arrival_latitude"]),
            "longitude": float(row["arrival_longitude"]),
            "altitude": float(row["arrival_altitude"]),
        }

# Add nodes (airports)
for iata, attrs in airports.items():
    G.add_node(iata, **attrs)

# Add edges (routes)
for _, row in df.iterrows():
    source = row["departure_iata"]
    target = row["arrival_iata"]

    flights_per_week = (
        int(row["flights_per_week"]) if pd.notna(row["flights_per_week"]) else 1
    )
    price = float(row["price"]) if pd.notna(row["price"]) else 0
    duration = int(row["common_duration"]) if pd.notna(row["common_duration"]) else 0
    if pd.notna(row["airline_names"]) and row["airline_names"].strip():
        airlines = [a.strip() for a in row["airline_names"].split(",")]
    else:
        airlines = []

    G.add_edge(
        source,
        target,
        flights_per_week=flights_per_week,
        price=price,
        duration=duration,
        airlines=airlines,
    )

In [ ]:
# Basic statistics
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()

print(f"Nodes (airports): {n_nodes}")
print(f"Edges (routes): {n_edges}")
print(f"Network type: Directed weighted graph")
print(f"Edge weights: Flights per week")
print(f"Additional attributes: Price, duration")

**Network visualization**

In [ ]:
def plot_european_airport_network(G, figsize=(16, 12), cmap="hsv", base_only=False):
    """
    Plots the European air transportation network.

    Parameters:
        G: The airline network
        figsize: Figure size.
        cmap: Colormap for nodes.
        base_only: If True, only draw base map (light gray nodes/edges) and return (fig, ax, pos).
    """

    # Positions (longitude, latitude)
    pos = {
        node: (G.nodes[node]["longitude"], G.nodes[node]["latitude"])
        for node in G.nodes()
    }

    # Node sizes proportional to weighted degree
    node_strength = dict(G.degree(weight="flights_per_week"))
    node_sizes = [node_strength[node] * 0.02 for node in G.nodes()]

    # Edge widths proportional to flights per week
    edge_widths = [G[u][v]["flights_per_week"] * 0.005 for u, v in G.edges()]

    # Node color based on total flights
    color_nodes = [
        sum(G[node][nbr]["flights_per_week"] for nbr in G.neighbors(node))
        for node in G.nodes()
    ]

    # Plot
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=ccrs.PlateCarree())

    # Automatic zoom
    lons = np.array([coord[0] for coord in pos.values()])
    lats = np.array([coord[1] for coord in pos.values()])
    buffer = 2
    lon_min = min(lons.min() - buffer, -30)
    lon_max = lons.max() + buffer
    lat_min = min(lats.min() - buffer, 25)
    lat_max = lats.max() + buffer
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    # Map background
    ax.add_feature(cfeature.LAND, facecolor="#f8f8f8")
    ax.add_feature(cfeature.OCEAN, facecolor="white")
    ax.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=0.2)
    countries = cfeature.NaturalEarthFeature(
        category="cultural",
        name="admin_0_countries",
        scale="10m",
        facecolor="none",
        edgecolor="#bbbbbb",
    )
    ax.add_feature(countries, linewidth=0.4)

    # Gridlines
    gl = ax.gridlines(
        draw_labels=True, linewidth=0.2, color="gray", alpha=0.15, linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False

    # Base network in light gray
    nx.draw_networkx_edges(
        G, pos, width=edge_widths, edge_color="black", alpha=0.15, arrows=False, ax=ax
    )

    nx.draw_networkx_nodes(
        G,
        pos,
        node_size=node_sizes,
        node_color=color_nodes,
        cmap=cmap,
        alpha=0.6,
        ax=ax,
    )

    if base_only:
        # Return the figure, axis, and positions so overlays can be added
        return fig, ax, pos

    nodes = nx.draw_networkx_nodes(
        G,
        pos,
        node_size=node_sizes,
        node_color=color_nodes,
        cmap=cmap,
        alpha=0.75,
        ax=ax,
    )
    cbar = plt.colorbar(nodes, ax=ax, shrink=0.7)
    cbar.set_label("Node weighted degree (total flights per week)")
    ax.set_title(
        "European Airport Network (weighted degree as node size)", fontweight="bold"
    )

    plt.show()

In [ ]:
plot_european_airport_network(G)

---
<a id='section3'></a>
## **3. Network characterization** 

<a id='section31'></a>
### **3.1 Macroscopic characterization** 

In [ ]:
def macroscopic_characterization(G):
    """
    Macroscopic characterization. Analyze global network properties.
    """
    # Convert to undirected for some metrics
    G_undirected = G.to_undirected()
    print("\n\033[1m1. Degree (Directed and unweighted graph):\033[0m")

    # In-degree and out-degree
    in_degrees = dict(G.in_degree(weight='flights_per_week'))
    out_degrees = dict(G.out_degree(weight='flights_per_week'))

    # Calculate Total Degree (or Strength) for each node: In + Out
    total_degrees = {
        node: in_degrees[node] + out_degrees[node] for node in G.nodes()}
    total_values = list(total_degrees.values())

    # Min / max nodes
    min_in_node = min(in_degrees, key=in_degrees.get)
    max_in_node = max(in_degrees, key=in_degrees.get)
    min_out_node = min(out_degrees, key=out_degrees.get)
    max_out_node = max(out_degrees, key=out_degrees.get)
    min_total_node = min(total_degrees, key=total_degrees.get)
    max_total_node = max(total_degrees, key=total_degrees.get)

    # Mean and median
    mean_in_degree = np.mean(list(in_degrees.values()))
    median_in_degree = statistics.median(in_degrees.values())
    mean_out_degree = np.mean(list(out_degrees.values()))
    median_out_degree = statistics.median(out_degrees.values())
    mean_total = np.mean(total_values)
    median_total = statistics.median(total_values)

    print("   In-degree (arrivals):")
    print(f"     - Minimum: {in_degrees[min_in_node]:.2f} (node {min_in_node})")
    print(f"     - Median:  {median_in_degree:.2f}")
    print(f"     - Average: {mean_in_degree:.2f}")
    print(f"     - Maximum: {in_degrees[max_in_node]:.2f} (node {max_in_node})")
    print("   Out-degree (departures):")
    print(f"     - Minimum: {out_degrees[min_out_node]:.2f} (node {min_out_node})")
    print(f"     - Median:  {median_out_degree:.2f}")
    print(f"     - Average: {mean_out_degree:.2f}")
    print(f"     - Maximum: {out_degrees[max_out_node]:.2f} (node {max_out_node})")
    print("   Total-degree (In-Degree + Out-Degree):")
    print(f"     - Minimum: {total_degrees[min_total_node]:.2f} (node {min_total_node})")
    print(f"     - Median:  {median_total:.2f}")
    print(f"     - Average: {mean_total:.2f}")
    print(f"     - Maximum: {total_degrees[max_total_node]:.2f} (node {max_total_node})")

    # 2. Density
    print("\n\033[1m2. Density:\033[0m")
    # Unweighted Density
    unweighted_density = nx.density(G)
    # Weighted Density (Average Capacity per potential edge)
    N = G.number_of_nodes()
    total_weight = G.size(weight="flights_per_week")
    weighted_density = total_weight / (N * (N - 1))

    print(f"     - Unweighted density (structural): {unweighted_density:.4f} ({unweighted_density * 100:.2f}%)")
    print(f"     - Weighted density (intensity): {weighted_density:.4f} flights/week/potential_route ({weighted_density * 100:.2f}%)")

    # 3. Clustering Coefficient: independent on weights, dependent on directionality
    # since it accounts for the proportion of possible triangles (triads)
    print("\n\033[1m 3. Clustering Coefficient (CC):\033[0m")

    print("    Directed and unweighted graph:")
    clustering_coeff_unwdir = nx.clustering(G)
    min_clustering_node_unwdir = min(
        clustering_coeff_unwdir, key=clustering_coeff_unwdir.get)
    average_clustering_unwdir = np.mean(list(clustering_coeff_unwdir.values()))
    max_clustering_node_unwdir = max(
        clustering_coeff_unwdir, key=clustering_coeff_unwdir.get)
    print(f"     - Minimum CC: {clustering_coeff_unwdir[min_clustering_node_unwdir]:.4f} (node {min_clustering_node_unwdir})")
    print(f"     - Average CC: {average_clustering_unwdir:.4f}")
    print(f"     - Maximum CC: {clustering_coeff_unwdir[max_clustering_node_unwdir]:.4f} (node {max_clustering_node_unwdir})")

    print("    Directed and Weighted Graph (flights per week):")
    clustering_coeff_wdir = nx.clustering(G, weight="flights_per_week")
    min_clustering_node_wdir = min(clustering_coeff_wdir, key=clustering_coeff_wdir.get)
    average_clustering_wdir = np.mean(list(clustering_coeff_wdir.values()))
    max_clustering_node_wdir = max(clustering_coeff_wdir, key=clustering_coeff_wdir.get)
    print(f"     - Minimum CC: {clustering_coeff_wdir[min_clustering_node_wdir]:.4f} (node {min_clustering_node_wdir})")
    print(f"     - Average CC: {average_clustering_wdir:.4f}")
    print(f"     - Maximum CC: {clustering_coeff_wdir[max_clustering_node_wdir]:.4f} (node {max_clustering_node_wdir})")

    # 4. Reciprocity (Accounting for direction)
    # Standard (Unweighted) Reciprocity
    unweighted_reciprocity = nx.reciprocity(G)
    # Weighted Reciprocity
    total_weight = G.size(weight="flights_per_week")
    mutual_weight = 0
    for u, v, data in G.edges(data=True):
        if G.has_edge(v, u):
            # We only count each pair once, so we check order to avoid double counting
            if u < v:
                w_uv = data.get("flights_per_week", 0)
                w_vu = G[v][u].get("flights_per_week", 0)
                mutual_weight += 2 * min(w_uv, w_vu)

    weighted_reciprocity = mutual_weight / total_weight

    print("\n\033[1m 4. Reciprocity (always Directed Graph):\033[0m")
    print(f"     - Unweighted graph: {unweighted_reciprocity:.4f}")
    print(f"     - Weighted reciprocity: {weighted_reciprocity:.4f}")
    
    if nx.is_strongly_connected(G):
        # 5. Average Path Length (on largest connected component)
        print(f"\nThe network is strongly connected.")
        print(f"\n\033[1m 5. Average Shortest Path Length:\033[0m")
        avg_path_length = nx.average_shortest_path_length(G)
        print(f"      - Directed and Unweighted Graph: {avg_path_length:.2f}")
        avg_path_length_w = nx.average_shortest_path_length(G, weight="duration")
        print(f"     - Directed and Weighted Graph (duration): {avg_path_length_w:.2f}")

        # 6. Diameter:
        diameter_du = nx.diameter(G())
        diameter_dw = nx.diameter(G(weight="duration"))

        print("\n\033[1m 6. Network diameter:\033[0m")
        print(f"     - Directed and Unweighted Graph: {diameter_du}")
        print(f"     - Directed and Weighted Graph (duration): {diameter_dw}")
    else:
        # Use largest strongly connected component
        print("\nThe network is not strongly connected. The largest strongly connected component (LSCC) is used.")
        largest_scc = max(nx.strongly_connected_components(G), key=len)
        print(f"LSCC size: {len(largest_scc)} nodes ({len(largest_scc) / G.number_of_nodes() * 100:.1f}%)")
        G_scc = G.subgraph(largest_scc)
        avg_path_length = nx.average_shortest_path_length(G_scc)
        print("\n\033[1m 5. Average Shortest Path Length (in LSCC):\033[0m")
        print(f"      - Directed and unweighted graph: {avg_path_length:.2f}")
        avg_path_length_w = nx.average_shortest_path_length(G_scc, weight="duration")
        print(f"      - Directed and weighted graph (duration): {avg_path_length_w:.2f}")

        # 6. Diameter:
        diameter_du = nx.diameter(G_scc)
        diameter_dw = nx.distance_measures.diameter(G_scc, weight="duration")

        print("\n\033[1m 6. Network diameter (in LSCC):\033[0m")
        print(f"     - Directed and unweighted graph: {diameter_du}")
        avg_path_length_w = nx.average_shortest_path_length(G_scc, weight="duration")
        print(f"     - Directed and weighted graph (duration): {avg_path_length_w:.2f}")

    # 7. Assortativity
    # Compute the 4 types of Assortativity for the Directed Graph
    # x = source node degree type, y = target node degree type
    in_in_assort = nx.degree_assortativity_coefficient(G, x="in", y="in")
    out_out_assort = nx.degree_assortativity_coefficient(G, x="out", y="out")
    out_in_assort = nx.degree_assortativity_coefficient(G, x="out", y="in")
    in_out_assort = nx.degree_assortativity_coefficient(G, x="in", y="out")

    in_in_assort_w = nx.degree_assortativity_coefficient(
        G, weight="flights_per_week", x="in", y="in")
    out_out_assort_w = nx.degree_assortativity_coefficient(
        G, weight="flights_per_week", x="out", y="out")
    out_in_assort_w = nx.degree_assortativity_coefficient(
        G, weight="flights_per_week", x="out", y="in")
    in_out_assort_w = nx.degree_assortativity_coefficient(
        G, weight="flights_per_week", x="in", y="out")
        
    print("\n\033[1m 7. Assortativity:\033[0m")
    print("    Directed and unweighted graph:")
    print(f"      - In-In degree correlation:   {in_in_assort:.4f}")
    print(f"      - Out-Out degree correlation: {out_out_assort:.4f}")
    print(f"      - Out-In degree correlation:  {out_in_assort:.4f}")
    print(f"      - In-Out degree correlation:  {in_out_assort:.4f}")
    print("    Directed and weighted graph (flights_per_week):")
    print(f"      - In-In degree correlation:   {in_in_assort_w:.4f}")
    print(f"      - Out-Out degree correlation: {out_out_assort_w:.4f}")
    print(f"      - Out-In degree correlation:  {out_in_assort_w:.4f}.")
    print(f"      - In-Out degree correlation:  {in_out_assort_w:.4f}")
    
macroscopic_characterization(G)

The **degree** of the network was measured using flights_per_week as the edges weight. In the results, the large gap between total median degree (72) and total average degree (526) highlights a _heavy-tailed degree distribution_, characteristic of _hub-and-spoke_ air transportation networks. A few major hubs (most notably AMS) concentrate a substantial fraction of weekly flights (7,431), while the majority of airports remain lightly connected. This structure enhances efficiency and reach but also implies high vulnerability to disruptions at key hubs.

The total minimum degree of 1 (from HVG) confirms the presence of peripheral airports. It also coincides with the minimum in-degree, but differenciates with the minimum out-degree of 0 (MLH), suggesting a sink node or data limitation. As for the maximum out-degree, AMS is recognized as the largest departure hub with a total of 3,798 destination routes, which reinforces its central operational role. The equality of average in- and out-degree (263) reflects global flow balance across the network.

Regarding the **density**, the unweighted value shows that structurally, the network is _sparse_, with limited direct connectivity between airports. Meanwhile, the weighted density shows that despite having few direct connections, the active routes are extremely high-frequency routes, showing high traffic concentration. A low value of unweighted density and a high weighted density is representative of a _Hub-and-spoke_ structure.

As for the **clustering coefficient**, the air transportation network exhibits high structural clustering (many local triangular connections) but very low weighted clustering, meaning that most clustered connections involve low-frequency flights, while high-traffic routes are more centralized and less locally interconnected (consistent with a hub-and-spoke network structure). The average weighted clustering value of 0.024 indicates that traffic-intensive directed triangles are uncommon.

**Reciprocity** measures the extent to which flight connections are bidirectional (i.e. how well balances the network is). In our directed and weighted air transportation network, reciprocity of value 0.8867 indicates that routes usually have both directions, which is the most common scenario for commercial air transportation. Also, it suggests that flight frequencies are not perfectly balanced between opposite directions, likely reflecting asymmetric demand, scheduling constraints and hub-related traffic imbalances.

The air transportation network is not strongly connected, meaning that not all airports are mutually reachable following flight directions. Therefore, analyses of distance-based metrics are restricted to the Largest Strongly Connected Component (LSCC), which contains 239 airports, representing 53.7% of the total network.

The **short average path length** of 1.86 shows a highly efficient topological structure. On average, any airport in the LSCC can be reached from another using fewer than two flights, reflecting strong hub connectivity and small-world characteristics. However, its larger value when duration (in minutes) is used as edge weights, indicates that, although few connections are needed, long-haul flights significantly increase effective distance in terms of time.

The **diameter** of 4 shows that the longest shortest path between any two airports in the LSCC requires at most four flights, confirming a compact and well-connected core network. Meanwhile, the weighted diameter represents the maximum minimum travel time between any two airports in the LSCC. A value of 211.75 minutes highlights the presence of very long-distance or low-frequency connections that define the temporal extremes of the network.

Overall, the contrast between duration-weighted and unweighted distances showcases the difference between structural efficiency (few transfers) and operational cost (total travel time).

The network's general **dissortativity** represents that airports tend to connect to other airports with different traffic levels: regardless of direction, airports with many arrivals or departures tend to connect to airports with fewer arrivals or departures. When weighting by flight frequency, assortativity pattern is still negative but weaker reflecting the same behaviour with lower strength. In particular, the Out–In correlation indicates that high-traffic source airports are somewhat more likely to connect to high-traffic destination airports.

In [ ]:
# Unweighted directed network:
# Degree sequence
degrees = np.array([G.degree(n) for n in G.nodes() if G.degree(n) > 0])
n = len(degrees)

# Linear histogram
degree_counts = Counter(degrees)
k = np.array(sorted(degree_counts.keys()))
pk = np.array([degree_counts[ki] / n for ki in k])

# Logarithmic binning
bins_log = 25
kmin, kmax = degrees.min(), degrees.max()
bins = np.logspace(np.log10(kmin), np.log10(kmax), bins_log + 1)
hist, bin_edges = np.histogram(degrees, bins=bins, density=False)

# Normalize to probability density
bin_widths = np.diff(bin_edges)
pk_log = hist / (n * bin_widths)
k_log = np.sqrt(bin_edges[1:] * bin_edges[:-1])

# Compute slope (γ)
mask = pk_log > 0
x = np.log10(k_log[mask])
y = np.log10(pk_log[mask])
slope, intercept, r_value, _, _ = linregress(x, y)
gamma = -slope
r2 = r_value ** 2

# Plot
network_title = "Degree Distribution (unweighted)"

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Linear histogram
ax[0].bar(k, pk, width=0.8, color="#ff7f00", edgecolor="black")
ax[0].set_xlabel("$k$", fontsize=10)
ax[0].set_ylabel("$P(k)$", fontsize=10)
ax[0].set_title(f"{network_title} - Linear histogram", fontsize=12, fontweight="bold")
ax[0].tick_params(axis="both", labelsize=12)

# Log–log histogram
ax[1].bar(k_log, pk_log, width=bin_widths, color="#ff7f00",
          edgecolor="black", align="center")
ax[1].set_xscale("log")
ax[1].set_yscale("log")
ax[1].set_xlabel("$k$", fontsize=10)
ax[1].set_ylabel("$P(k)$", fontsize=10)
ax[1].set_title(f"{network_title} - Log-log histogram", fontsize=12, fontweight="bold")
ax[1].tick_params(axis="both", labelsize=12)
ax[1].set_xlim(bin_edges[0], bin_edges[-1])

fig.tight_layout()
plt.show()

print(f"{network_title}: γ = {gamma:.3f}, R² = {r2:.3f}")

# Weighted directed network:
# Strenght sequence
strengths = np.array([G.degree(n, weight="flights_per_week") for n in G.nodes() if G.degree(n) > 0])
n = len(strengths)

# Linear histogram
strength_counts = Counter(strengths)
s = np.array(sorted(strength_counts.keys()))
ps = np.array([strength_counts[ki] / n for ki in s])

# Logarithmic binning
bins_log = 25
smin, smax = strengths.min(), strengths.max()
bins = np.logspace(np.log10(smin), np.log10(smax), bins_log + 1)
hist, bin_edges = np.histogram(strengths, bins=bins, density=False)

# Normalize to probability density
bin_widths = np.diff(bin_edges)
ps_log = hist / (n * bin_widths)
s_log = np.sqrt(bin_edges[1:] * bin_edges[:-1])

# Compute slope (γ)
mask = ps_log > 0
x = np.log10(s_log[mask])
y = np.log10(ps_log[mask])
slope, intercept, r_value, _, _ = linregress(x, y)
gamma = -slope
r2 = r_value ** 2

# Plot
network_title = "Degree Distribution (weighted)"

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Linear histogram
ax[0].bar(s, ps, width=0.8, color="#ff7f00", edgecolor="black")
ax[0].set_xlabel("$s$", fontsize=10)
ax[0].set_ylabel("$P(s)$", fontsize=10)
ax[0].set_title(f"{network_title} - Linear histogram", fontsize=12, fontweight="bold")
ax[0].tick_params(axis="both", labelsize=12)

# Log–log histogram
ax[1].bar(s_log, ps_log, width=bin_widths, color="#ff7f00",
          edgecolor="black", align="center")
ax[1].set_xscale("log")
ax[1].set_yscale("log")
ax[1].set_xlabel("$s$", fontsize=10)
ax[1].set_ylabel("$P(s)$", fontsize=10)
ax[1].set_title(f"{network_title} - Log-log histogram", fontsize=12, fontweight="bold")
ax[1].tick_params(axis="both", labelsize=12)
ax[1].set_xlim(bin_edges[0], bin_edges[-1])

fig.tight_layout()
plt.show()

print(f"{network_title}: γ = {gamma:.3f}, R² = {r2:.3f}")

Degree distributions of the air transportation network show heavy-tailed behavior in both unweighted (structural/connectivity) and weighted (traffic intensity) cases. Most airports have few connections or flights, while a small number of hubs concentrate disproportionately many links and high traffic volumes. Log-log histograms reveal approximate linearity, consistent with a power-law distribution, with exponents γ = 1.084 (unweighted) and γ = 1.022 (weighted), indicating an extreme hub-dominated, scale-free–like network.

The network exhibits a hub-and-spoke structure:
- Structural hubs: airports with many connections
- Traffic hubs: airports with large volumes of flights

Such heterogeneity has important implications for network robustness, as the system is expected to be resilient to random failures but vulnerable to targeted disruptions affecting hub airports.

**Route frequency distribution:**

In [ ]:
weights = [d["flights_per_week"] for _, _, d in G.edges(data=True)]

plt.hist(weights, bins=10, edgecolor="white")
plt.xlabel("Flights per week")
plt.ylabel("Number of routes")
plt.title("Distribution of route frequency", fontweight="bold")
plt.show()

The distribution of route frequencies reveals a heterogeneous pattern, with a large number of routes operating at high weekly frequencies (one to twenty flights per week), alongside a substantial fraction of low-frequency connections. This reflects the coexistence of highly stable, frequently operated routes (typically linking major hubs and less frequent connections associated with regional or peripheral airports). Such heterogeneity in edge weights highlights the importance of considering weighted network measures when analyzing traffic flow, robustness, and failure propagation.

<a id='section32'></a>
### **3.2 Microscopic characterization** 

In [ ]:
def microscopic_characterization(G):
    """
    Microscopic characterization. Analyze individual node properties and centrality measures
    """
    # 1. Degree Centrality
    print("\n\033[1m1. Top 5 Hubs by Degree Centrality (unweighted):\033[0m")
    degree_centrality = nx.degree_centrality(G)
    top_degree = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:5]

    print(f"\n    {'Rank':<6}{'IATA':<8}{'Airport':<40}{'City':<20}{'Degree':<10}")
    print("    "+"-" * 82)
    for i, (node, cent) in enumerate(top_degree, 1):
        attrs = G.nodes[node]
        degree = G.degree(node)
        print(f"    {i:<6}{node:<8}{attrs['name'][:38]:<40}{attrs['city'][:18]:<20}{degree:<10}")

    # 2. Betweenness Centrality
    print("\n\033[1m2. Top 5 Airports by Betweenness Centrality:\033[0m")
    print("    (Measuring importance as transit/bridge nodes)")

    betweenness = nx.betweenness_centrality(G, weight="flights_per_week")
    top_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]

    print(f"\n    {'Rank':<6}{'IATA':<8}{'Airport':<40}{'Betweenness':<15}")
    print("    "+"-" * 69)
    for i, (node, cent) in enumerate(top_betweenness, 1):
        attrs = G.nodes[node]
        print(f"    {i:<6}{node:<8}{attrs['name'][:38]:<40}{cent:<15.6f}")

    # 3. Closeness Centrality (on largest connected component)
    print("\n\033[1m3. Top 5 Airports by Closeness Centrality (duration-weighted):\033[0m")
    print("   (Measuring average distance to all other nodes)")
    
    largest_scc = max(nx.strongly_connected_components(G), key=len)
    G_scc = G.subgraph(largest_scc).copy()

    try:
        closeness = nx.closeness_centrality(G, distance='duration')
        top_closeness = sorted(closeness.items(), key=lambda x: x[1], reverse=True)[:5]

        print(f"\n    {'Rank':<6}{'IATA':<8}{'Airport':<40}{'Closeness':<15}")
        print("    "+"-" * 69)
        for i, (node, cent) in enumerate(top_closeness, 1):
            attrs = G.nodes[node]
            print(f"    {i:<6}{node:<8}{attrs['name'][:38]:<40}{cent:<15.6f}")
    except:
        print("     Network is not strongly connected - computing on largest component")
        closeness = nx.closeness_centrality(G_scc, distance='duration')
        top_closeness = sorted(closeness.items(), key=lambda x: x[1], reverse=True)[:5]

        print(f"\n    {'Rank':<6}{'IATA':<8}{'Airport':<40}{'Closeness':<15}")
        print("-" * 69)
        for i, (node, cent) in enumerate(top_closeness, 1):
            attrs = G.nodes[node]
            print(f"    {i:<6}{node:<8}{attrs['name'][:38]:<40}{cent:<15.6f}")

    # 4. Eigenvector Centrality (on largest connected component)
    print("\n\033[1m4. Top 10 Airports by Eigenvector Centrality:\033[0m")
    print("    (Measuring influence based on connections to important nodes)")

    # Extract largest connected component
    if G.is_directed():
        largest_cc = max(nx.weakly_connected_components(G), key=len)
    else:
        largest_cc = max(nx.connected_components(G), key=len)

    G_lcc = G.subgraph(largest_cc)

    # Compute eigenvector centrality
    eigenvector = nx.eigenvector_centrality(G_lcc, max_iter=1000, weight="flights_per_week")

    top_eigenvector = sorted(eigenvector.items(), key=lambda x: x[1], reverse=True)[:5]

    print(f"\n    {'Rank':<6}{'IATA':<8}{'Airport':<40}{'Eigenvector':<15}")
    print("    "+"-" * 69)

    for i, (node, cent) in enumerate(top_eigenvector, 1):
        attrs = G_lcc.nodes[node]
        print(f"    {i:<6}{node:<8}{attrs['name'][:38]:<40}{cent:<15.6f}")
  
     # 5. PageRank Centrality (on largest connected component)
    print("\n\033[1m5. Top 5 Airports by PageRank Centrality:\033[0m")

    # Compute eigenvector centrality
    pagerank = nx.pagerank(G, weight="flights_per_week")

    eigenvector = nx.eigenvector_centrality(G_lcc, max_iter=1000)

    top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]

    print(f"\n    {'Rank':<6}{'IATA':<8}{'Airport':<40}{'PageRank':<15}")
    print("    "+"-" * 69)

    for i, (node, cent) in enumerate(top_pagerank, 1):
        attrs = G.nodes[node]
        print(f"    {i:<6}{node:<8}{attrs['name'][:38]:<40}{cent:<15.6f}")

    # 6. In-degree vs Out-degree Analysis
    in_degrees_weighted = dict(G.in_degree(weight="flights_per_week"))
    out_degrees_weighted = dict(G.out_degree(weight="flights_per_week"))

    print("\n\033[1m6. Directional Flow Analysis:\033[0m")

    # Top destinations (highest weighted in-degree)
    top_destinations = sorted(in_degrees_weighted.items(), key=lambda x: x[1], reverse=True)[:5]
    print("   Top 5 Destinations (highest weighted in-degree):")
    for node, in_deg in top_destinations:
        print(f"     - {node} ({G.nodes[node]['city']}): {in_deg} incoming routes")

    # Top origins (highest weighted out-degree)
    top_origins = sorted(out_degrees_weighted.items(), key=lambda x: x[1], reverse=True)[:5]
    print("   Top 5 Origins (highest weighted out-degree):")
    for node, out_deg in top_origins:
        print(f"     - {node} ({G.nodes[node]['city']}): {out_deg} outgoing routes")
        
microscopic_characterization(G)

Amsterdam Schiphol (AMS) functions as the network’s operational anchor. Leading in both Eigenvector (0.275) and PageRank (0.022), AMS possesses the highest concentration of influence, meaning its importance is derived from its connections to other top-tier global hubs (with massive physical flow), as evidenced by its rank as the #1 destination and origin with over 7,400 combined weekly flight units. While other airports may have more unique routes, AMS is the airport with high-value, high-frequency traffic across the continent.

In contrast, London Stansted (STN) represents the network’s "geographic reach". By having the highest Unweighted Degree Centrality (319 routes), it maintains the most diverse array of unique destinations in Europe. However, its absence from the top of the Eigenvector and Closeness lists suggests a "point-to-point" strategy characteristic of low-cost carriers. Stansted focuses on connecting a vast quantity of secondary cities that larger hubs ignore, serving as a massive "utility" airport that prioritizes the width of the network over the intensity or status of individual connections.

A critical finding is the emergence of geographic gatekeepers identified through Betweenness Centrality. Stockholm (ARN) and Oslo (OSL) lead this category (0.031 and 0.029 respectively), indicating they are the most indispensable "bridges" in the system. Because of their peripheral location, these airports are essential transit points; removing them would disproportionately disrupt paths between Northern Europe and the rest of the continent. Similarly, the appearance of Gran Canaria (LPA) and Corfu (CFU) in this list highlights how seasonal or island-based nodes can become vital bottlenecks that control the flow of traffic to specific regions.

Closeness results, weighted by duration, shift the focus toward the "efficiency core" of Central Europe. Munich (MUC) and Vienna (VIE) dominate this metric, proving to be the most accessible nodes for the entire network. Unlike the sheer volume seen in Amsterdam or the unique connections in Stansted, these airports offer the lowest average "temporal distance" to all other points in Europe. This makes them the operational heart of the continent ideal for time-sensitive transfers and logistics because they are, mathematically and geographically, the most "reachable" destinations for every other airport in the system.

Finally, PageRank centrality reveals a highly centralized European air transportation network dominated by a small set of major hubs. Amsterdam Schiphol (AMS) ranks first (0.0223), confirming its critical role as the most influential airport, strongly connected to other important nodes. Frankfurt (FRA), Barcelona (BCN), Madrid (MAD), and Munich (MUC) follow closely, all functioning as key international gateways. The relatively small differences in PageRank among the top five indicate a tightly connected core rather than a single monopoly hub. While this structure enhances efficiency and accessibility, it also increases vulnerability, as disruptions at these airports could have widespread network effects.

In [ ]:
def plots_microscopic_char(pos):
    weighted_degree = dict(G.degree(weight="flights_per_week"))
    eigenvector_centralities = nx.eigenvector_centrality(G, weight="flights_per_week")
    betweenness_centralities = nx.betweenness_centrality(G, weight="flights_per_week")
    page_rank_centralities = nx.pagerank(G, weight="flights_per_week")
    closeness_centralities = nx.closeness_centrality(G, distance="flights_per_week")

    centralities = [
        ("degree", weighted_degree),
        ("eigenvector", eigenvector_centralities),
        ("betweenness", betweenness_centralities),
        ("PageRank", page_rank_centralities),
        ("closeness", closeness_centralities)]

    n_cols = 2
    n_rows = int(np.ceil(len(centralities) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
    axes = axes.flatten()

    node_strength = dict(G.degree(weight="flights_per_week"))
    node_sizes = [node_strength[node] * 0.02 for node in G.nodes()]
    edge_widths = [G[u][v]["flights_per_week"] * 0.005 for u, v in G.edges()]
    
    for i, (name, centrality_dict) in enumerate(centralities):
        color_nodes = [centrality_dict[node] for node in G.nodes()]

        nx.draw_networkx(
            G,
            pos=pos,
            node_size=node_sizes,
            node_color=color_nodes,
            edge_color="grey",
            width=edge_widths,
            cmap="Reds",
            with_labels=False,
            ax=axes[i])

        axes[i].set_title(
            f"Weighted {name.capitalize()}", fontsize=15, fontweight="bold")
        axes[i].axis("off")

    for j in range(len(centralities), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

pos = {node: (G.nodes[node]["longitude"], G.nodes[node]["latitude"]) for node in G.nodes()}

plots_microscopic_char(pos)

The figures compare five centrality metrics on a geographic network, illustrating how "importance" shifts based on the chosen centrality measure:

1. **Degree & PageRank:** Highlight high-traffic local hubs. They appear similar, identifying a few dominant nodes with the most immediate connections.
2. **Eigenvector:** Concentrates influence within a dense central core. Nodes are valued for being connected to other powerful nodes, creating a group of high-ranking neighbors.
3. **Betweenness:** Identifies critical "brokers" or transit points. Unlike the others, it highlights isolated nodes that act as unique bridges between different regions.
4. **Closeness:** Displays the most widespread activity. Because it measures the average distance to all other points, the entire central "mass" of the network appears significant, as these nodes are most accessible to the whole system.

The visualization demonstrates that while some hubs are universally important, others only emerge as vital when considering specific roles like efficiency (Closeness) or control (Betweenness).

**Network stratification by structural role**

In [ ]:
degrees = dict(G.degree())
weighted_degree = dict(G.degree(weight="flights_per_week"))

betweenness = nx.betweenness_centrality(G, weight="flights_per_week")
pagerank = nx.pagerank(G, weight="flights_per_week")

centrality_df = pd.DataFrame(
    {"degree": degrees,
     "weighted_degree": weighted_degree,
     "betweenness": betweenness,
     "pagerank": pagerank}
).sort_values("betweenness", ascending=False)

# Percentile thresholds
wd_90 = centrality_df["weighted_degree"].quantile(0.90)
bt_90 = centrality_df["betweenness"].quantile(0.90)

print(f"Weighted degree quantile 90%: {wd_90:.4f}.")
print(f"Weighted betweenness quantile 90%: {bt_90:.4f}.")

In [ ]:
# Assign an structural layer to each airport
def assign_layer(row):
    if row["weighted_degree"] >= wd_90:
        return "Core hubs"
    elif row["betweenness"] >= bt_90:
        return "Bridge nodes"
    else:
        return "Peripheral nodes"

centrality_df["layer"] = centrality_df.apply(assign_layer, axis=1)

# Check the layer distribution
print(centrality_df["layer"].value_counts())

In [ ]:
# Show examples by layer
display(centrality_df.groupby("layer").head(5))

In [ ]:
# Stratification visualization
centrality_df["layer"].value_counts().plot(kind="bar")
plt.ylabel("Number of airports")
plt.title("Network stratification by structural role", fontweight="bold")
plt.show()

Based on centrality measures, the European air transportation network exhibits a clear stratified organization. Airports can be grouped into three distinct structural layers according to their functional roles within the network.

The majority of airports belong to a peripheral layer, characterized by low degree and low weighted degree. These airports handle limited traffic and rely on connections to more central nodes to access the rest of the network. Despite their large number, their removal has a relatively small impact on global connectivity, which contributes to the network’s robustness against random failures.

A second layer consists of core hub airports, identified by high weighted degree values. These airports concentrate a substantial fraction of the total traffic and play a critical role in maintaining the operational capacity of the system. Disruptions affecting this layer lead to a rapid degradation of network connectivity, as observed in the targeted attack scenarios.

Finally, a smaller group of bridge airports exhibits high betweenness centrality despite having moderate degree and traffic volume. These nodes act as structural intermediaries, connecting peripheral regions to the network core. Although they are not major traffic hubs, their strategic position makes them essential for maintaining efficient connectivity across different regions.

This hierarchical stratification highlights the coexistence of traffic concentration and structural mediation within the European air transportation network and provides a structural explanation for the robust-but-fragile behavior observed in the robustness analysis.

<a id='section33'></a>
### **3.3 Community detection** 

In [ ]:
def community_detection(G):
    """
    Detect communities using Louvain, Leiden, Infomap, and SBM.
    Plots community size distributions and network maps with legends.
    """
    results = {}
    G_undirected = G.to_undirected()

    # 1. Louvain
    try:
        louvain_partition = community_louvain.best_partition(
            G_undirected, weight="flights_per_week")
        louvain_sizes = Counter(louvain_partition.values())
        results["Louvain"] = {"partition": louvain_partition, "sizes": louvain_sizes}
    except ImportError:
        print("Louvain not installed, skipping Louvain method")
        louvain_partition = None

    # 2. Leiden
    try:
        # Convert NetworkX to igraph
        mapping = {node: i for i, node in enumerate(G_undirected.nodes())}
        inv_mapping = {i: node for node, i in mapping.items()}
        edges = [(mapping[u], mapping[v]) for u, v in G_undirected.edges()]
        g = ig.Graph(edges, directed=False)
        # Edge weights
        weights = [
            G_undirected[u][v].get("flights_per_week", 1)
            for u, v in G_undirected.edges()]
        partition = la.find_partition(g, la.ModularityVertexPartition, weights=weights)
        leiden_partition = {
            inv_mapping[i]: comm for i, comm in enumerate(partition.membership)}
        leiden_sizes = Counter(leiden_partition.values())
        results["Leiden"] = {"partition": leiden_partition, "sizes": leiden_sizes}
    except ImportError:
        print("Leiden or igraph not installed, skipping Leiden method")
        leiden_partition = None

    # 3. Infomap (flow-based)
    try:
        # Build numeric network for Infomap
        G_numeric, mapping_df = build_numeric_network(df)

        # Mapping from numeric ID to airport code
        inv_mapping = {
            row["numeric_id"]: row["airport_iata"] for _, row in mapping_df.iterrows()}

        # Initialize Infomap (two-level, directed)
        infomap = Infomap("--two-level --directed")

        # Add links using numeric IDs
        for u, v, data in G_numeric.edges(data=True):
            w = float(data.get("flights_per_week", 1))
            infomap.add_link(u, v, w)

        # Run Infomap
        infomap.run()

        # Map back numeric IDs to airport codes
        infomap_partition = {
            inv_mapping[node_id]: module
            for node_id, module in infomap.get_modules().items()}

        # Count communities
        infomap_sizes = Counter(infomap_partition.values())
        results["Infomap"] = {"partition": infomap_partition, "sizes": infomap_sizes}

    except ImportError:
        print("Infomap not installed, skipping Infomap")
        infomap_partition = None

    # 4. SBM (Spectral)
    sbm_results = stochastic_block_model_detection(G_undirected)
    if sbm_results:
        sbm_partition = sbm_results["communities_dict"]
        sbm_sizes = Counter(sbm_partition.values())
        results["SBM"] = {"partition": sbm_partition, "sizes": sbm_sizes}

    for method, data in results.items():
        plot_community_sizes(
            data["sizes"], title=f"{method} Community Size Distribution")

    return results

def build_numeric_network(df):
    """
    Build a directed weighted network with numeric node IDs.

    Returns:
        G_numeric: Directed weighted graph with numeric nodes
        mapping_df: Mapping between numeric ID and airport IATA codes
    """
    # Assign numeric IDs to airports
    airports = {}
    numeric_id = 0
    for _, row in df.iterrows():
        dep_iata = row["departure_iata"]
        arr_iata = row["arrival_iata"]

        for iata in [dep_iata, arr_iata]:
            if iata not in airports:
                airports[iata] = numeric_id
                numeric_id += 1

    # Create mapping DataFrame
    mapping_df = (
        pd.DataFrame(
            {"numeric_id": list(airports.values()),
             "airport_iata": list(airports.keys())})
        .sort_values("numeric_id")
        .reset_index(drop=True))

    # Build numeric graph
    G_numeric = nx.DiGraph()

    # Add nodes with numeric ID
    for _, row in mapping_df.iterrows():
        G_numeric.add_node(row["numeric_id"], airport_iata=row["airport_iata"])

    # Add edges with numeric IDs and weights
    for _, row in df.iterrows():
        source = airports[row["departure_iata"]]
        target = airports[row["arrival_iata"]]
        flights_per_week = (
            int(row["flights_per_week"]) if pd.notna(row["flights_per_week"]) else 1)
        price = float(row["price"]) if pd.notna(row["price"]) else 0
        duration = (
            int(row["common_duration"]) if pd.notna(row["common_duration"]) else 0)

        G_numeric.add_edge(
            source,
            target,
            flights_per_week=flights_per_week,
            price=price,
            duration=duration)

    n_nodes = G_numeric.number_of_nodes()
    n_edges = G_numeric.number_of_edges()

    return G_numeric, mapping_df

def stochastic_block_model_detection(G, n_communities=None):
    """
    Detect communities using Stochastic Block Model (SBM) approximation via Spectral Clustering.
    G is a undirected network graph.
    """
    try:
        # Adjacency matrix
        adj_matrix = nx.to_numpy_array(G)

        # Estimate number of communities using eigengap heuristic if not specified
        if n_communities is None:
            laplacian = nx.laplacian_matrix(G).astype(float)
            n_nodes = G.number_of_nodes()
            k = min(20, n_nodes - 2)
            try:
                eigenvalues = eigsh(
                    laplacian, k=k, which="SM", return_eigenvectors=False)
                eigenvalues = np.sort(eigenvalues)
                eigengaps = np.diff(eigenvalues)
                n_communities = np.argmax(eigengaps) + 2
                n_communities = max(2, min(n_communities, 15))
            except:
                n_communities = 5

        print(f"   Using {n_communities} communities for SBM")

        # Spectral Clustering
        spectral = SpectralClustering(
            n_clusters=n_communities,
            affinity="precomputed",
            assign_labels="kmeans",
            random_state=42)
        labels = spectral.fit_predict(adj_matrix)

        # Convert to dictionary: node -> community
        communities_dict = {node: int(label) for node, label in zip(G.nodes(), labels)}

        # Convert to list of sets per community
        communities_list = [set() for _ in range(n_communities)]
        for node, label in communities_dict.items():
            communities_list[label].add(node)
        communities_list = [c for c in communities_list if len(c) > 0] 

        # Compute modularity
        modularity = nx.community.modularity(G, communities_list)

        return {
            "communities_dict": communities_dict,
            "communities_list": communities_list,
            "n_communities": len(communities_list),
            "modularity": modularity}

    except ImportError:
        print("   scikit-learn not installed, skipping SBM")
        return None

def plot_community_sizes(sizes_counter, title="Community Size Distribution"):
    """
    Plot Community Size Distribution.
    """
    sizes_sorted = sorted(sizes_counter.values(), reverse=True)
    plt.figure(figsize=(8, 4))
    plt.bar(
        range(1, len(sizes_sorted) + 1),
        sizes_sorted,
        color="#377eb8",
        edgecolor="black",
        alpha=0.7)
    plt.xlabel("Community Rank")
    plt.ylabel("Community Size")
    plt.title(title, fontsize=12, fontweight="bold")
    plt.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()

def draw_community_map(G, partition, pos=None, title="Community Map"):
    """
    Draw Network Colored by Communities with Legend
    """
    if pos is None:
        pos = nx.spring_layout(G, seed=42, weight="flights_per_week")
    plt.figure(figsize=(12, 10))
    communities = set(partition.values())
    n_communities = len(communities)
    
    if n_communities < 4:
        chosen_cmap = "winter"
    else:
        chosen_cmap = "gist_rainbow"
        
    cmap = plt.cm.get_cmap(chosen_cmap, n_communities)
    
    sorted_communities = sorted(communities)
    
    for i, community_id in enumerate(sorted_communities):
        nodes = [n for n in partition if partition[n] == community_id]
        
        # Use black for the last community if n_communities >= 3
        if n_communities >= 3 and i == len(sorted_communities) - 1:
            node_color = 'black'
        else:
            node_color = [cmap(i)]
        
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=nodes,
            node_size=50,
            node_color=node_color,
            label=f"Community {community_id}")
    
    # Weighted edges
    weights = np.array([G[u][v].get("flights_per_week", 1) for u, v in G.edges()])
    if len(weights) > 0:
        weights = 0.5 + 3 * (weights / weights.max())
    else:
        weights = 0.5
    nx.draw_networkx_edges(G, pos, width=weights, alpha=0.3, edge_color="gray")
    plt.title(title, fontsize=15, fontweight="bold")
    plt.axis("off")
    plt.legend(scatterpoints=1)
    plt.tight_layout()
    plt.show()
    
results = community_detection(G)

In [ ]:
G_undirected = G.to_undirected()
# Positions (longitude, latitude)
pos = {
    node: (G_undirected.nodes[node]["longitude"], G_undirected.nodes[node]["latitude"])
    for node in G.nodes()}
for method, data in results.items():
    draw_community_map(G, data["partition"], pos, title=f"{method} Community Map")

The provided community detection maps illustrate how the European air transportation network can be partitioned into distinct functional groups or regions based on flight connectivity. The different algorithms (Louvain, Leiden, Infomap, and SBM) reveal varying granularities of how the airspace is structured.

**Regional Clustering (Louvain & Leiden):** Both the Louvain and Leiden algorithms identify roughly 5–6 major communities that largely align with European geography. They highlight distinct clusters for Northern Europe (green), the Mediterranean/Iberian Peninsula (red), and Central/Western Europe (yellow/purple).

**Hierarchical Stability (Infomap):** The Infomap results show a massive, dominant central community (red) that encompasses the majority of the continent, with smaller, highly specific clusters in Scandinavia (yellow) and isolated peripheral nodes. This suggests that the core of the European network is so tightly integrated that it functions as a single giant component.

**Structural Simplification (SBM):** The Stochastic Block Model (SBM) provides the most simplified view, essentially dividing the network into a massive "core" and a few peripheral "outlier" nodes. This highlights the "Hub-and-Spoke" nature of the continent, where most airports belong to a singular, dense block of high connectivity.

When cross-referenced with the previous centrality data, it becomes clear that airports like Amsterdam (AMS) and Frankfurt (FRA) aren't just important individually; they serve as the "anchor points" for the largest communities identified here. Conversely, airports with high Betweenness like Stockholm (ARN) are the leaders of their own smaller, specialized communities (the green/yellow clusters in the north).

---
<a id='section4'></a>
## **4. Network sparsification**  

In this section, we apply sparsification techniques. Sparsification aims to remove less significant connections while retaining the backbone of the network that governs its main connectivity and flow patterns. In the context of air transportation, this allows us to highlight the most important routes such as those with high traffic or strategic relevance, while maintaining the overall topology and functional properties of the network. 

In [ ]:
# Print some network info
print(f"European air transportation network:")
print(f"  - Number of nodes: {G.number_of_nodes()}")
print(f"  - Number of edges: {G.number_of_edges()}")
print(
    f"  - Number of weakly connected components: {nx.number_weakly_connected_components(G)}"
)
print(
    f"  - Number of strongly connected components: {nx.number_strongly_connected_components(G)}"
)

# Keep the largest weakly connected component
largest_wcc = max(nx.weakly_connected_components(G), key=len)
G_lcc = G.subgraph(largest_wcc).copy()

print(f"\nLargest weakly connected component:")
print(f"  - Number of nodes: {G_lcc.number_of_nodes()}")
print(f"  - Number of edges: {G_lcc.number_of_edges()}")
print(
    f"  - Fraction of original nodes: {G_lcc.number_of_nodes() / G.number_of_nodes():.3f}"
)
print(
    f"  - Fraction of original edges: {G_lcc.number_of_edges() / G.number_of_edges():.3f}"
)

<a id='section41'></a>
### **4.1 Weights thresholding**  

First, we applied the weights thresholding method using ``flights_per_week`` as the edge weight. In this method, less frequently used routes (lower ``flight_per_week`` values) are progressively removed, allowing us to examine how the network’s connectivity changes as weaker links are eliminated. By analyzing the number and size of weakly and strongly connected components at each removal fraction, we can assess the robustness of the network and identify the subset of routes that are essential for the network.

In [ ]:
edges_data = []

for u, v, data in G_lcc.edges(data=True):
    edges_data.append(
        {
            "source": u,
            "target": v,
            "flights_per_week": data["flights_per_week"],
            "duration": data["duration"],
            "price": data["price"],
        }
    )

edges_df = pd.DataFrame(edges_data)

edges_df_freq = edges_df.copy()
edges_df_freq = edges_df_freq.sort_values(
    by="flights_per_week", ascending=True
).reset_index(drop=True)

fraction_removed = np.linspace(0, 0.8, 41)
number_weak_components_freq = []
number_strong_components_freq = []
size_lwcc_freq = []
size_lscc_freq = []
number_edges_original = G_lcc.number_of_edges()

for frac in fraction_removed:
    number_edges_to_remove = int(number_edges_original * frac)
    edges_to_remove = edges_df_freq.iloc[0:number_edges_to_remove]
    G_temp = G_lcc.copy()
    G_temp.remove_edges_from(zip(edges_to_remove["source"], edges_to_remove["target"]))

    n_weak_components = nx.number_weakly_connected_components(G_temp)
    n_strong_components = nx.number_strongly_connected_components(G_temp)
    number_weak_components_freq.append(n_weak_components)
    number_strong_components_freq.append(n_strong_components)

    if n_weak_components > 0:
        largest_wcc = max(nx.weakly_connected_components(G_temp), key=len)
        size_lwcc_freq.append(len(largest_wcc))
    else:
        size_lwcc_freq.append(0)

    if n_strong_components > 0:
        largest_scc = max(nx.strongly_connected_components(G_temp), key=len)
        size_lscc_freq.append(len(largest_scc))
    else:
        size_lscc_freq.append(0)

In [ ]:
# Plot for weak connectivity
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].plot(
    fraction_removed,
    number_weak_components_freq,
    marker="s",
    color="#377eb8",
    label="Remove low frequency",
)
ax[0].set_xlabel("Fraction of edges removed", fontsize=10)
ax[0].set_ylabel("Number of weak components", fontsize=10)
ax[0].set_title(
    "Weak connectivity fragmentation (by flight frequency)",
    fontsize=12,
    fontweight="bold",
)
ax[0].grid(alpha=0.3)

ax[1].plot(fraction_removed, size_lwcc_freq, marker="s", color="#377eb8")
ax[1].set_xlabel("Fraction of edges removed", fontsize=10)
ax[1].set_ylabel("Size of largest weak component", fontsize=10)
ax[1].set_title("LWCC size (by flight frequency)", fontsize=12, fontweight="bold")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot for strong connectivity
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].plot(
    fraction_removed,
    number_strong_components_freq,
    marker="s",
    color="#377eb8",
    label="Remove low frequency",
)
ax[0].set_xlabel("Fraction of edges removed", fontsize=10)
ax[0].set_ylabel("Number of strong components", fontsize=10)
ax[0].set_title(
    "Strong connectivity fragmentation (by flight frequency)",
    fontsize=12,
    fontweight="bold",
)
ax[0].grid(alpha=0.3)

ax[1].plot(fraction_removed, size_lscc_freq, marker="s", color="#377eb8")
ax[1].set_xlabel("Fraction of edges removed", fontsize=10)
ax[1].set_ylabel("Size of largest strong component", fontsize=10)
ax[1].set_title("LSCC size (by flight frequency)", fontsize=12, fontweight="bold")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Find critical points where network fragments significantly
def find_critical_point(components_list, threshold=5):
    for i, n_comp in enumerate(components_list):
        if n_comp > threshold:
            return fraction_removed[i]
    return None


critical_weak_freq = find_critical_point(number_weak_components_freq)
critical_strong_freq = find_critical_point(number_strong_components_freq, threshold=10)

print("\nCritical weak fragmentation points (>5 components):")
print(
    f"  - Weights threshold (frequency): {critical_weak_freq:.2%}"
    if critical_weak_freq
    else "  - Weights threshold (frequency): Not reached"
)

print("\nCritical strong fragmentation points (>10 components):")
print(
    f"  - Weights threshold (frequency): {critical_strong_freq:.2%}"
    if critical_strong_freq
    else "  - Weights threshold (frequency): Not reached"
)

From the plots, we can observe that as we remove edges with low flight frequency (low ``flights_per_week``), the network starts to fragment. The number of Weakly Connected Components (WCCs) increases gradually at first, then jumps sharply when key connections are removed. Additionally, we can see that weak connectivity fragmentation has a steeper slope than strong connectivity fragementation. This means that it is more fragile to edge removals.

The point where the curve rises above 5 WCCs corresponds to the critical weak fragmentation point that we established. The critical points for weak connectivity indicate that removing just 6% of edges based on ``flights_per_week`` is enough to fragment the network into more than five weakly connected components.   

Strong connectivity, on the other hand, remains relatively resilient at these low levels of sparsification, as the network never reaches the threshold of more than ten Strongly Connected Components (SCCs). 

<a id='section42'></a>
### **4.2 Disparity filter**  

Afterwards, we also applied the disparity filter which evaluates the statistical significance of each route relative to the overall traffic distribution of its connected airports, identifying edges that carry disproportionately high weights compared to what would be expected by chance. By progressively removing edges with high p-values (statistically insignificant), we can analyze how the network’s connectivity (both weakly and strongly connected components) changes while retaining the most structurally and functionally important routes. This approach identifies the most important routes in the network, while taking into account that different airports have different levels of traffic.

In [ ]:
edges_df_disparity = edges_df.copy()

# Compute out-strength and in-strength
out_strength = dict(G_lcc.out_degree(weight="flights_per_week"))
in_strength = dict(G_lcc.in_degree(weight="flights_per_week"))
out_degree = dict(G_lcc.out_degree())
in_degree = dict(G_lcc.in_degree())

# Map to edges df
edges_df_disparity["source_out_strength"] = edges_df_disparity["source"].map(
    out_strength
)
edges_df_disparity["source_out_degree"] = edges_df_disparity["source"].map(out_degree)
edges_df_disparity["target_in_strength"] = edges_df_disparity["target"].map(in_strength)
edges_df_disparity["target_in_degree"] = edges_df_disparity["target"].map(in_degree)

# Compute disparity filter p-values
# p_out: probability that weight is random given source's out-distribution
# p_in: probability that weight is random given target's in-distribution
edges_df_disparity["p_out"] = (
    1
    - edges_df_disparity["flights_per_week"] / edges_df_disparity["source_out_strength"]
) ** (edges_df_disparity["source_out_degree"] - 1)

edges_df_disparity["p_in"] = (
    1
    - edges_df_disparity["flights_per_week"] / edges_df_disparity["target_in_strength"]
) ** (edges_df_disparity["target_in_degree"] - 1)

# Take minimum p-value (edge is significant if significant in either direction)
edges_df_disparity["p"] = np.minimum(
    edges_df_disparity["p_out"], edges_df_disparity["p_in"]
)

# Sort by p-value
# Remove edges with high p-values
edges_df_disparity = edges_df_disparity.sort_values(
    by="p", ascending=False
).reset_index(drop=True)

number_weak_components_disparity = []
number_strong_components_disparity = []
size_lwcc_disparity = []
size_lscc_disparity = []

for frac in fraction_removed:
    number_edges_to_remove = int(number_edges_original * frac)
    edges_to_remove = edges_df_disparity.iloc[0:number_edges_to_remove]
    G_temp = G_lcc.copy()
    G_temp.remove_edges_from(zip(edges_to_remove["source"], edges_to_remove["target"]))

    n_weak_components = nx.number_weakly_connected_components(G_temp)
    n_strong_components = nx.number_strongly_connected_components(G_temp)
    number_weak_components_disparity.append(n_weak_components)
    number_strong_components_disparity.append(n_strong_components)

    if n_weak_components > 0:
        largest_wcc = max(nx.weakly_connected_components(G_temp), key=len)
        size_lwcc_disparity.append(len(largest_wcc))
    else:
        size_lwcc_disparity.append(0)

    if n_strong_components > 0:
        largest_scc = max(nx.strongly_connected_components(G_temp), key=len)
        size_lscc_disparity.append(len(largest_scc))
    else:
        size_lscc_disparity.append(0)

In [ ]:
# Compare disparity filter with weight thresholding (weak connectivity)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].plot(
    fraction_removed,
    number_weak_components_freq,
    marker="s",
    color="#377eb8",
    label="Weight Threshold",
    linestyle="--",
)
ax[0].plot(
    fraction_removed,
    number_weak_components_disparity,
    marker="o",
    color="#ff7f00",
    label="Disparity Filter",
)
ax[0].set_xlabel("Fraction of edges removed", fontsize=10)
ax[0].set_ylabel("Number of weak components", fontsize=10)
ax[0].set_title(
    "Weak connectivity fragmentation comparison", fontsize=12, fontweight="bold"
)
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(
    fraction_removed,
    size_lwcc_freq,
    marker="s",
    color="#377eb8",
    label="Weight Threshold",
    linestyle="--",
)
ax[1].plot(
    fraction_removed,
    size_lwcc_disparity,
    marker="o",
    color="#ff7f00",
    label="Disparity Filter",
)
ax[1].set_xlabel("Fraction of edges removed", fontsize=10)
ax[1].set_ylabel("Size of largest weak component", fontsize=10)
ax[1].set_title("LWCC size comparison", fontsize=12, fontweight="bold")
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare disparity filter with weight thresholding (strong connectivity)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].plot(
    fraction_removed,
    number_strong_components_freq,
    marker="s",
    color="#377eb8",
    label="Weight Threshold",
    linestyle="--",
)
ax[0].plot(
    fraction_removed,
    number_strong_components_disparity,
    marker="o",
    color="#ff7f00",
    label="Disparity Filter",
)
ax[0].set_xlabel("Fraction of edges removed", fontsize=10)
ax[0].set_ylabel("Number of strong components", fontsize=10)
ax[0].set_title(
    "Strong connectivity fragmentation comparison", fontsize=12, fontweight="bold"
)
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(
    fraction_removed,
    size_lscc_freq,
    marker="s",
    color="#377eb8",
    label="Weight Threshold",
    linestyle="--",
)
ax[1].plot(
    fraction_removed,
    size_lscc_disparity,
    marker="o",
    color="#ff7f00",
    label="Disparity Filter",
)
ax[1].set_xlabel("Fraction of edges removed", fontsize=10)
ax[1].set_ylabel("Size of largest strong component", fontsize=10)
ax[1].set_title("LSCC size comparison", fontsize=12, fontweight="bold")
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
critical_weak_disparity = find_critical_point(number_weak_components_disparity)
critical_strong_disparity = find_critical_point(
    number_strong_components_disparity, threshold=10
)

print("\nCritical weak fragmentation points (>5 components):")
print(
    f"  - Disparity filter: {critical_weak_disparity:.2%}"
    if critical_weak_disparity
    else "  - Disparity filter: Not reached"
)

print("\nCritical strong fragmentation points (>10 components):")
print(
    f"  - Disparity filter: {critical_strong_disparity:.2%}"
    if critical_strong_disparity
    else "  - Disparity filter: Not reached"
)

From the plots, we can observe that as we remove edges with high p-values, the network starts to fragment. The number WCCs increases gradually at first, then jumps sharply when key connections are removed. As for the weights thresholding, we can see that weak connectivity fragmentation has a steeper slope than strong connectivity fragementation, meaning that it is more fragile to edge removals.

In the case of the disparity filter, the critical points for weak connectivity indicate that removing just 2% is enough to fragment the network into more than five weakly connected components.   

Strong connectivity, on the other hand, remains relatively resilient at these low levels of sparsification, as the network never reaches the threshold of more than ten SCCs. 

In [ ]:
print("\nSparsification efficiency at 25% edge removal:")
idx_50 = int(len(fraction_removed) * 0.25 / 0.8)

print("\nWeak connectivity:")
print(
    f"  - Weights threshold (frequency): {size_lwcc_freq[idx_50]} nodes, {number_weak_components_freq[idx_50]} weak components"
)
print(
    f"  - Disparity filter: {size_lwcc_disparity[idx_50]} nodes, {number_weak_components_disparity[idx_50]} weak components"
)

print("\nStrong connectivity:")
print(
    f"  - Weights threshold (frequency): {size_lscc_freq[idx_50]} nodes, {number_strong_components_freq[idx_50]} strong components"
)
print(
    f"  - Disparity filter: {size_lscc_disparity[idx_50]} nodes, {number_strong_components_disparity[idx_50]} strong components"
)

When examining the effects of removing 25% of edges, both sparsification methods lead to a substantial fragmentation: the LWCC retains around 424-429 nodes, but the network breaks into roughly 22-17 weak components, while strong connectivity shows an even greater impact, leaving 237-238 nodes in the LSCC and more than 200 strong components in total. These results suggest that the network is structurally robust in terms of node coverage (most airports remain reachable in some way), but directional connectivity between airports is highly fragile, and both sparsification methods produce significant fragmentation at high levels of edge removal. 

So from the analysis, we can obtain the routes that are less important/essential:
- Based on weights threshold

In [ ]:
# Based on weights threshold (first 20 lowest-frequency flights)
edges_df_freq.iloc[0:10]

- Based on disparity filter

In [ ]:
# Based on disparity filter (first 20 highest p values)
edges_df_disparity.sort_values("p", ascending=False).head(10)[
    ["source", "target", "flights_per_week", "duration", "price", "p"]
]

To identify important/essential routes we could: 

- Look for those edges that are key for network cohesion

In [ ]:
# Suppose at 25% fraction removed, weak connectivity fragments
number_edges_to_remove = int(number_edges_original * 0.25)
critical_edges = edges_df_freq.iloc[number_edges_to_remove - 5 : number_edges_to_remove]
critical_edges

- Analyze betweenness centrality: high betweenness means that the edge lies on many shortest paths so removing them could disrupt connectivity

In [ ]:
# Compute edge betweenness centrality
edge_bc = nx.edge_betweenness_centrality(G_lcc, weight="flights_per_week")

# Sort and get top 10
top_edges = sorted(edge_bc.items(), key=lambda x: x[1], reverse=True)[:10]
df_top_edges = pd.DataFrame(top_edges, columns=["edge", "betweenness_centrality"])

df_top_edges[["source", "target"]] = pd.DataFrame(
    df_top_edges["edge"].tolist(), index=df_top_edges.index
)
df_top_edges = df_top_edges[["source", "target", "betweenness_centrality"]]
display(df_top_edges)

---
<a id='section5'></a>
## **5. Robustness and percolation analysis** 

Robustness and percolation analysis evaluate a network's ability to maintain its structural integrity and core functionality as nodes or links are removed. By using percolation theory, the degradation of the Largest Strongest Connected Component (the largest cluster of interconnected nodes) can be tracked until the system reaches a critical threshold where it collapses into isolated fragments. This approach is essential for identifying "percolation thresholds", the tipping points at which an infrastructure system like an aviation network loses its ability to facilitate long-distance travel.

In [ ]:
# Functions that support the percolation analysis
def largest_scc_fraction(G, N):
    """Relative size of largest strongly connected component"""
    if G.number_of_nodes() == 0 or G.number_of_edges() == 0:
        return 0.0
    scc = max(nx.strongly_connected_components(G), key=len)
    return len(scc) / N

def network_efficiency(G):
    """Global efficiency"""
    if G.number_of_nodes() < 2:
        return 0.0
    Gu = G.to_undirected()
    if not nx.is_connected(Gu):
        Gu = Gu.subgraph(max(nx.connected_components(Gu), key=len))
    return nx.global_efficiency(Gu)

def average_path_length(G):
    """APL on largest SCC"""
    if G.number_of_edges() == 0:
        return np.nan
    scc = max(nx.strongly_connected_components(G), key=len)
    Gs = G.subgraph(scc)
    if Gs.number_of_nodes() < 2:
        return np.nan
    return nx.average_shortest_path_length(Gs, weight="duration")

def run_removal_process(G_original, node_order, label, results):
    """
    Generic percolation process
    """
    G = G_original.copy()
    N = G_original.number_of_nodes()
    removed = 0

    results[label] = {"p": [], "lscc": [], "efficiency": [], "apl": []}

    for node in node_order:
        p = removed / N
        results[label]["p"].append(p)
        results[label]["lscc"].append(largest_scc_fraction(G, N))
        results[label]["efficiency"].append(network_efficiency(G))
        results[label]["apl"].append(average_path_length(G))

        if G.has_node(node):
            G.remove_node(node)
            removed += 1

    print(f"Percolation completed: {label}")
                
# Create an empty results dictionary
results = defaultdict(list)

<a id='section51'></a>
### **5.1 Random failure**

Simulates unpredictable disruptions, such as local equipment failures or weather delays. Scale-free networks (as our European air transportation nertwork) are remarkably robust against those scenarios, as most nodes have few connections and their loss barely impacts the overall connectivity.

In [ ]:
def random_failure(G_original, results, seed=42):
    """
    Random node removal (robustness to random failures)
    """
    np.random.seed(seed)
    nodes = list(G_original.nodes())
    np.random.shuffle(nodes)
    run_removal_process(G_original, nodes, "Random Failure", results)

# Random failures
random_failure(G, results)

<a id='section52'></a>
### **5.2 Targeted attack (degree)** 

Eliminates the most connected "hubs" first. This strategy is highly destructive to aviation systems, as removing a few high-degree airports (like Stansted or Amsterdam) causes the network to collapse rapidly into disconnected clusters.

**Directed strength Attacks (flights per week):**

In [ ]:
def targeted_attack_in_strength(G_original, results):
    """
    Targeted attack: highest in-strength (flights_per_week) first
    """
    # Calculate in-degree weighted by flights
    strengths = dict(G_original.in_degree(weight="flights_per_week"))
    nodes = sorted(strengths, key=strengths.get, reverse=True)
    run_removal_process(G_original, nodes, "Targeted Attack (In-Strength)", results)


def targeted_attack_out_strength(G_original, results):
    """
    Targeted attack: highest out-strength (flights_per_week) first
    """
    # Calculate out-degree weighted by flights
    strengths = dict(G_original.out_degree(weight="flights_per_week"))
    nodes = sorted(strengths, key=strengths.get, reverse=True)
    run_removal_process(G_original, nodes, "Targeted Attack (Out-Strength)", results)


def targeted_attack_total_strength(G_original, results):
    """
    Targeted attack: highest total strength (flights_per_week) first
    """
    # G.degree(weight=...) on a directed graph returns In + Out strength
    strengths = dict(G_original.degree(weight="flights_per_week"))
    nodes = sorted(strengths, key=strengths.get, reverse=True)
    run_removal_process(G_original, nodes, "Targeted Attack (Total Strength)", results)

targeted_attack_in_strength(G, results)
targeted_attack_out_strength(G, results)
targeted_attack_total_strength(G, results)

<a id='section53'></a>
### **5.3 Targeted attack (betweenness)** 

Targets the "bridges" or gatekeepers that facilitate shortest paths. By removing high-betweenness nodes (like Stockholm or Athens), attackers can maximize the average distance between all remaining nodes, effectively paralyzing the network’s efficiency and flow control.

In [ ]:
def targeted_attack_weighted_betweenness(G_original, results):
    """
    Targeted attack: highest duration-weighted betweenness first
    """
    betweenness = nx.betweenness_centrality(G_original, weight="duration")
    nodes = sorted(betweenness, key=betweenness.get, reverse=True)
    run_removal_process(
        G_original, nodes, "Targeted Attack (Duration-Betweenness)", results
    )

targeted_attack_weighted_betweenness(G, results)

<a id='section54'></a>
### **5.4 Results of percolation analysis:** 

In [ ]:
def plot_percolation_curves(results):
    """
    Plot percolation and robustness curves
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    for label, data in results.items():
        p = data["p"]
        axes[0, 0].plot(p, data["lscc"], label=label)
        axes[0, 1].plot(p, data["efficiency"], label=label)
        axes[1, 0].plot(p, data["apl"], label=label)

    axes[0, 0].set_title("Percolation: LSCC Size", fontweight="bold")
    axes[0, 0].set_xlabel("Fraction of Nodes Removed (p)")
    axes[0, 0].set_ylabel("LSCC / N")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].set_title("Network Efficiency", fontweight="bold")
    axes[0, 1].set_xlabel("Fraction of Nodes Removed (p)")
    axes[0, 1].set_ylabel("Efficiency (undirected)")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].set_title("Average Path Length (LSCC)", fontweight="bold")
    axes[1, 0].set_xlabel("Fraction of Nodes Removed (p)")
    axes[1, 0].set_ylabel("APL")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].axis("off")

    plt.tight_layout()
    plt.savefig("robustness_percolation_analysis.png", dpi=300)
    plt.show()

def estimate_percolation_threshold(results, method="lscc", threshold=0.1):
    """
    Estimate critical percolation threshold p_c
    """
    print("Percolation threshold estimation:")

    for label, data in results.items():
        values = data["lscc"] if method == "lscc" else data["efficiency"]
        p_vals = data["p"]

        for p, v in zip(p_vals, values):
            if v <= threshold:
                print(f"    - {label}: p_c ≈ {p:.3f}")
                break
                

# Visualization
plot_percolation_curves(results)

# Estimate critical thresholds
estimate_percolation_threshold(results, method="lscc", threshold=0.1)

The percolation analysis evaluates the structural integrity and robustness of the European air transportation network by simulating various node-removal strategies. By monitoring the degradation of the Largest Strongly Connected Component (LSCC), we can identify the "percolation threshold", which is the critical fraction of node removals required to cause a complete system collapse. The results clearly distinguish between the network's high tolerance for stochastic disruptions and its extreme vulnerability to strategic, targeted attacks on its primary hubs and bridges.

In Random Failure, the network exhibits remarkable resilience, maintaining a significant LSCC even as a large majority of nodes are removed. The percolation threshold for random failure is estimated at , meaning nearly 80% of airports must fail before the network loses global connectivity. This behavior is characteristic of scale-free networks, where most nodes have low connectivity; their random removal does not typically disrupt the critical "highways" that hold the broader system together.

Conversely, Targeted Attack, whether based on Degree (Strength) or Betweenness, cause the network to fragment rapidly. Attacks targeting in-strength, out-strength, or total strength, all result in system failure after removing only about 30% of the nodes. Targeted attacks on duration-weighted betweenness are similarly devastating, highlighting that the removal of critical gatekeepers and transit bridges is just as disruptive as the removal of high-volume hubs.

The impact of these targeted strategies is further evidenced by the dramatic spikes in Average Path Length (APL). As the most efficient bridges are removed, the "topological distance" between remaining cities increases sharply—peaking between  and —before the network finally splinters into isolated clusters. Additionally, while Network Efficiency declines steadily under random failure, it collapses abruptly in targeted scenarios, proving that the European aviation grid relies heavily on a small core of vital nodes for its operational performance.

---
<a id='section6'></a>
## **6. Diffusion dynamics** 

To study dynamical processes on the network, we analyze diffusion dynamics. In this context, diffusion models the spread of entities such as passengers, information, or disruptions through the air transportation system, where airports act as nodes and flight routes as transmission pathways. The directed and weighted nature of the network allows diffusion to be influenced by the intensity of connections, such as flight frequency or travel duration. This approach provides insight into how network structure and connectivity patterns affect the propagation of flows across the network

<a id='section61'></a>
### **6.1 Single random walk**  

Single walks are more useful for tracking individual passenger journeys or example paths. We will analyze:
- **Simple random walk:** At each airport, it chooses the next flight uniformly at random from all outgoing flights.
- **Weighted random walk:** Chooses the next flight weighted by flights_per_week

In [ ]:
def simple_random_walk(G, start_node, steps=50):
    """
    Performs a simple random walk on the network.
    At each step, it randomly chooses one of the outgoing edges with equal probability.

    Parameters:
        G: The airline network
        start_node: Starting airport IATA code
        steps: Number of steps to walk
    """
    if start_node not in G.nodes():
        raise ValueError(f"Start node {start_node} not in graph")

    path = [start_node]
    current = start_node

    for _ in range(steps):
        neighbors = list(G.successors(current))

        # No outgoing edges
        if not neighbors:
            break

        # Randomly choose next node with uniform probability
        next_node = random.choice(neighbors)
        path.append(next_node)
        current = next_node

    return path

In [ ]:
def weighted_random_walk(G, start_node, steps=50, weight_attr="flights_per_week"):
    """
    Perform a weighted random walk where edge selection probability
    is proportional to the number of flights per week.

    Parameters:
        G: The airline network
        start_node: Starting airport IATA code
        steps: Number of steps to walk
        weight_attr: Edge attribute to use as weight (default: 'flights_per_week')
    """
    if start_node not in G.nodes():
        raise ValueError(f"Start node {start_node} not in graph")

    path = [start_node]
    current = start_node

    for _ in range(steps):
        neighbors = list(G.successors(current))

        # No outgoing edges
        if not neighbors:
            break

        # Get weights for each neighbor
        weights = []
        for neighbor in neighbors:
            weight = G[current][neighbor].get(weight_attr, 1)
            weights.append(weight)

        # Normalize weights to probabilities
        total_weight = sum(weights)
        probabilities = [w / total_weight for w in weights]

        # Choose next node based on weighted probabilities
        next_node = random.choices(neighbors, weights=probabilities)[0]
        path.append(next_node)
        current = next_node

    return path

In [ ]:
# Example with BCN
random.seed(42)
simple_walk_path = simple_random_walk(G, "BCN", steps=10)
weighted_walk_path = weighted_random_walk(G, "BCN", steps=10)

print(f"Simple walk path (first 10 airports):")
print(" -> ".join(simple_walk_path[:11]))

print(f"\nWeighted walk path (first 10 airports):")
print(" -> ".join(weighted_walk_path[:11]))

In [ ]:
def random_walk_gif(
    G, walk, output_filename="random_walk_map.gif", duration=10, figsize=(14, 10)
):
    """
    Create a GIF of a random walk.

    Parameters:
        G: The airline network
        walk: List of nodes representing the random walk path.
        output_filename: Name of the output GIF file.
        duration: Seconds per frame.
        figsize: Figure size for each frame.
    """

    filenames = []
    visited_nodes = set()
    visited_edges = []

    for i, node in enumerate(walk):
        prev = walk[i - 1] if i > 0 else None
        visited_nodes.add(node)
        if prev is not None:
            visited_edges.append((prev, node))

        # Base map and network
        fig, ax, pos = plot_european_airport_network(
            G, figsize=figsize, cmap="Greys", base_only=True
        )

        # Visited edges
        if visited_edges:
            nx.draw_networkx_edges(
                G,
                pos,
                edgelist=visited_edges,
                width=1,
                edge_color="darkblue",
                alpha=0.4,
                arrows=True,
                arrowsize=8,
                ax=ax,
            )

        # Highlight current edge
        if prev is not None:
            nx.draw_networkx_edges(
                G,
                pos,
                edgelist=[(prev, node)],
                width=4,
                edge_color="orange",
                arrows=True,
                arrowsize=20,
                ax=ax,
            )

        # Visited nodes
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=list(visited_nodes),
            node_color="blue",
            node_size=100,
            alpha=0.8,
            ax=ax,
        )

        # Highlight previous node
        if prev is not None:
            nx.draw_networkx_nodes(
                G,
                pos,
                nodelist=[prev],
                node_color="yellow",
                node_size=250,
                alpha=0.8,
                ax=ax,
            )

        # Highlight current node
        nx.draw_networkx_nodes(
            G, pos, nodelist=[node], node_color="red", node_size=500, ax=ax
        )

        # Label current node
        labels = nx.draw_networkx_labels(
            G, pos, labels={node: node}, font_size=12, font_weight="bold", ax=ax
        )

        for txt in labels.values():
            txt.set_zorder(8)

        ax.set_title(
            f"Random Walk - Step {i}/{len(walk) - 1}\nCurrent Airport: {node}"
            + (f" (from {prev})" if prev else " (START)"),
            fontsize=16,
            fontweight="bold",
        )

        plt.tight_layout()

        # Save frame
        fname = f"figures/geo_frame_{i:04d}.png"
        plt.savefig(fname, dpi=100, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        filenames.append(fname)

    # Create GIF
    frames = [PILImage.open(f) for f in filenames]
    frames[0].save(
        output_filename,
        save_all=True,
        append_images=frames[1:],
        duration=int(duration * 100),  # milliseconds
        loop=0,
    )  # infinite loop

    # Cleanup temporary frame files
    for fname in filenames:
        os.remove(fname)

    return output_filename

In [ ]:
# Create GIF with geographic layout
gif_file_simple = random_walk_gif(
    G,
    simple_walk_path,
    output_filename="figures/bcn_simple_random_walk.gif",
    duration=10,
    figsize=(14, 10))

gif_file_weighted = random_walk_gif(
    G,
    weighted_walk_path,
    output_filename="figures/bcn_weighted_random_walk.gif",
    duration=10,
    figsize=(14, 10))

print(f"GIF saved as: {os.path.basename(gif_file_simple)} (in 'figures' folder)")
print(f"GIF saved as: {os.path.basename(gif_file_weighted)} (in 'figures' folder)")

In [ ]:
# Display the Simple random walk GIF
display(IPImage(filename=gif_file_simple))

In [ ]:
# Display the Weighted random walk GIF
display(IPImage(filename=gif_file_weighted))

<a id='section62'></a>
### **6.2 Multiple random walks**  

Multiple walks approximate overall network traffic patterns (i.e., which airports are likely to be visited frequently)

To determine an adequate number of random walks, we analyzed the convergence of node visit frequencies as the number of walks increased.

In [ ]:
# Parameters
random.seed(42)
start_node = "BCN"
max_walks = 5000
steps = 50
check_points = [100, 500, 1000, 2000, 5000]

# Storage
convergence_data = []
visit_counts = Counter()

# Run random walks and track convergence
for i in range(1, max_walks + 1):
    path = weighted_random_walk(G, start_node, steps=steps)
    visit_counts.update(path)

    if i in check_points:
        total_visits = sum(visit_counts.values())
        top_10 = visit_counts.most_common(10)

        convergence_data.append(
            {
                "num_walks": i,
                "unique_nodes": len(visit_counts),
                "top_node": top_10[0][0] if top_10 else None,
                "top_node_freq": top_10[0][1] / total_visits if top_10 else 0,
            }
        )

# Convert to df
df = pd.DataFrame(convergence_data)

# Plot
fig, ax1 = plt.subplots(figsize=(8, 5))

# Plot top node frequency
ax1.plot(
    df["num_walks"],
    df["top_node_freq"],
    marker="s",
    color="tab:blue",
    label="Top airport frequency",
)
ax1.set_xlabel("Number of walks")
ax1.set_ylabel("Top airport frequency", color="#377eb8")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.set_ylim(0, df["top_node_freq"].max() + 0.05)

# Secondary y-axis: unique nodes visited
ax2 = ax1.twinx()
ax2.plot(
    df["num_walks"],
    df["unique_nodes"],
    marker="o",
    color="tab:orange",
    label="Unique airports visited",
)
ax2.set_ylabel("Unique airports visited", color="#ff7f00")
ax2.tick_params(axis="y", labelcolor="tab:orange")
ax2.set_ylim(0, G.number_of_nodes() + 5)

plt.title("Convergence of Weighted Random Walks from BCN", fontweight="bold")
fig.tight_layout()
plt.show()

The blue curve shows how the top airport’s relative frequency stabilizes. While the orange curve shows how the network gets covered. Initially, each new walk discovers new airports until it eventually plateaus as most nodes have been visited.  

Both the most visited airport and the number of unique nodes stabilized after approximately 1000 walks. Therefore, all subsequent analyses use 1000 random walks.

In [ ]:
def multiple_random_walks(
    G, start_node, num_walks=1000, steps=50, walk_type="weighted", **kwargs
):
    """
    Perform multiple random walks and aggregate statistics.

    Parameters:
        G: The airline network
        start_node: Starting airport IATA code
        num_walks: Number of random walks to perform
        steps: Steps per walk
        walk_type: Type of walk: 'simple' or 'weighted'
        **kwargs: Additional arguments for specific walk types
    """
    visit_counts = Counter()
    all_paths = []

    # Select walk function
    if walk_type == "simple":
        walk_func = simple_random_walk
    elif walk_type == "weighted":
        walk_func = weighted_random_walk
    else:
        raise ValueError(f"Unknown walk type: {walk_type}")

    # Perform walks
    for _ in range(num_walks):
        path = walk_func(G, start_node, steps=steps, **kwargs)
        all_paths.append(path)
        visit_counts.update(path)

    # Calculate statistics
    total_visits = sum(visit_counts.values())
    visit_frequencies = {
        node: count / total_visits for node, count in visit_counts.items()
    }

    # Get unique nodes visited
    unique_nodes = set(visit_counts.keys())
    coverage = len(unique_nodes) / G.number_of_nodes() * 100

    results = {
        "visit_counts": visit_counts,
        "visit_frequencies": visit_frequencies,
        "unique_nodes_visited": len(unique_nodes),
        "coverage_percentage": coverage,
        "all_paths": all_paths,
        "avg_path_length": np.mean([len(path) for path in all_paths]),
        "start_node": start_node,
    }

    return results

In [ ]:
# Multiple weighted random walks
random.seed(42)

results = multiple_random_walks(
    G, "BCN", num_walks=1000, steps=50, walk_type="weighted"
)

print(f"Results for multiple weighted random walks:")
print(f"  - Unique airports visited: {results['unique_nodes_visited']}")
print(f"  - Network coverage: {results['coverage_percentage']:.2f}%")
print(f"  - Average walk length: {results['avg_path_length']:.2f} steps")

# Top 10 most visited airports
top_10 = sorted(results["visit_frequencies"].items(), key=lambda x: x[1], reverse=True)[
    :10
]

top_10_data = []
for rank, (airport, freq) in enumerate(top_10, start=1):
    airport_name = G.nodes[airport].get("name", "Unknown")
    airport_city = G.nodes[airport].get("city", "Unknown")
    top_10_data.append(
        {
            "Rank": rank,
            "Airport": airport,
            "City": airport_city,
            "Visit Frequency (%)": freq * 100,
            "Name": airport_name,
        }
    )

df_top_10 = pd.DataFrame(top_10_data)

print("\nTop 10 most visited airports from BCN:")
display(df_top_10)

In [ ]:
# Plot
visit_freq = results["visit_frequencies"]
top_airports = sorted(visit_freq.items(), key=lambda x: x[1], reverse=True)[:10]

# Separate airport codes and frequencies
airports = [airport for airport, freq in top_airports]
frequencies = [freq * 100 for airport, freq in top_airports]

# Create horizontal bar chart
plt.figure(figsize=(8, 6))
bars = plt.barh(airports, frequencies, color="#377eb8")

# Highlight the start node if it's in the top airports
if results["start_node"] in airports:
    start_idx = airports.index(results["start_node"])
    bars[start_idx].set_color("#ff7f00")

# Labels and title
plt.xlabel("Visit frequency (%)")
plt.ylabel("Airport IATA code")
plt.title(
    f"Top {10} Most Visited Airports\nStarting from {results['start_node']}",
    fontweight="bold",
)
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

As we can see from the results obtained above, the number of unique airports visited is 402. This means that out of all airports in the network, the weighted random walks reached most of them (~90%).

Usually, weighted walks tend to favor hubs, but with 1000 walks, even smaller airports are reached indirectly. This shows that the network is highly connected, and the random walker can reach almost all nodes starting from BCN.  

Even though each walk could go up to 50 steps, the average is lower (~27). This happens because walks sometimes hit dead ends or nodes with fewer outgoing flights, or because repeated visits to high-degree nodes concentrate probability, effectively shortening the “exploration” of unique nodes. Weighted walks naturally loop around hubs, so a walker revisits major airports instead of moving far into peripheral airports.  

From the top 10 most visited airports from BCN, we can conclude that weighted random walks prioritize high-frequency routes and hubs. The presence of BCN at the top makes sense as starting nodes have loops and immediate flights back (local connectivity).

<a id='section63'></a>
### **6.3 Time Dependent Occupation Probability**  

Starting from Barcelona (BCN), we performed 1000 weighted random walks of up to 50 steps, simulating how a passenger might traverse the network over successive flights. From these walks, we computed the time-dependent occupation probability, which represents the likelihood of the passenger being at each airport at each step.

In [ ]:
# Parameters
num_steps = 50
all_paths = results["all_paths"]
nodes = list(G.nodes())
N = len(nodes)
node_index = {node: i for i, node in enumerate(nodes)}

# Initialize occupation probability matrix
# p_t[step, node] = probability of being at node at that step
p_t = np.zeros((num_steps + 1, N))

# Count visits at each step
for path in all_paths:
    for t, node in enumerate(path):
        if t > num_steps:
            break
        p_t[t, node_index[node]] += 1

# Normalize by number of walks to get probability
p_t /= len(all_paths)

# Select top 10 airports
top_airports = [airport for airport, freq in top_10]

plt.figure(figsize=(12, 5))
for airport in top_airports:
    i = node_index[airport]
    plt.plot(p_t[:, i], label=f"{airport} ({G.nodes[airport]['city']})", alpha=0.8)

plt.xlabel("Step")
plt.ylabel("Occupation probability")
plt.title("Time-dependent Occupation Probability for Top Airports", fontweight="bold")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Show df for top 10 airports
df_p_t = pd.DataFrame(p_t, columns=nodes)

df_p_t.insert(0, "Step", range(p_t.shape[0]))

top_airports_only = ["Step"] + top_airports
df_top_p_t = df_p_t[["Step"] + top_airports]
display(df_top_p_t.head(10))

The table shows the time dependent occupation probabilities for the top 10 most visited airports starting from Barcelona (BCN) over the first 10 steps of the random walks. Each row corresponds to a step in the walk, and each column gives the probability that a passenger is at a specific airport at that step. At step 0, the probability is 100% at BCN because all walks start there, and 0% for all other airports. From step 1 onward, the probabilities spread across the network according to the flight connections: airports directly connected to BCN, such as Madrid (MAD), Amsterdam (AMS), and Paris (CDG), receive nonzero probabilities early on. As steps increase, the probabilities fluctuate and gradually spread through the network, reflecting how passengers diffuse from the origin across Europe’s airline network.

In [ ]:
def occupation_probability_gif(
    G,
    p_t,
    steps=None,
    output_filename="occupation_prob.gif",
    figsize=(16, 12),
    cmap="hsv",
):
    """
    Create a GIF showing the diffusion of occupation probability over time.

    Parameters:
        G: networkx graph
        p_t: time-dependent occupation probability matrix (shape: [steps+1, N])
        steps: list of steps to include in GIF (default: all)
        output_filename: output GIF filename
        figsize: figure size
        cmap: colormap for nodes
    """

    if steps is None:
        steps = list(range(p_t.shape[0]))

    nodes = list(G.nodes())
    N = len(nodes)
    filenames = []

    for t in steps:
        # Base map
        fig, ax, pos = plot_european_airport_network(
            G, base_only=True, cmap="Greys", figsize=figsize
        )

        # Node sizes and colors based on probability at this step
        probs = p_t[t, :]
        node_sizes = [5000 * probs[i] for i in range(N)]
        node_colors = probs

        nx.draw_networkx_nodes(
            G,
            pos,
            node_size=node_sizes,
            node_color=node_colors,
            cmap=plt.cm.get_cmap(cmap),
            alpha=0.8,
            ax=ax,
        )

        # Colorbar
        sm = plt.cm.ScalarMappable(cmap=plt.cm.get_cmap(cmap))
        sm.set_array(probs)
        cbar = plt.colorbar(sm, ax=ax, shrink=0.7)
        cbar.set_label("Occupation probability")

        fig.subplots_adjust(left=0.05, right=0.99, top=0.99, bottom=0.01)

        # Title
        ax.set_title(
            f"Occupation Probability - Step {t}", fontsize=16, fontweight="bold"
        )

        # Save frame
        fname = f"figures/occupation_frame_{t:04d}.png"
        plt.savefig(fname, dpi=100, facecolor="white")
        plt.close(fig)
        filenames.append(fname)

    # Create GIF
    frames = [PILImage.open(f) for f in filenames]
    frames[0].save(
        output_filename,
        save_all=True,
        append_images=frames[1:],
        duration=2000,  # milliseconds
        loop=0,
    )

    # Cleanup temporary frame files
    for f in filenames:
        os.remove(f)

    return output_filename

In [ ]:
# Generate GIF
gif_file_occupation = occupation_probability_gif(
    G,
    p_t,
    steps=range(0, p_t.shape[0], 2),  # every 2 steps to reduce number of frames
    output_filename="figures/bcn_occupation_prob.gif",
    figsize=(14, 10),)

print(f"GIF saved as: {os.path.basename(gif_file_occupation)} (in 'figures' folder)")

In [ ]:
# Display the GIF
display(IPImage(filename=gif_file_occupation))

The GIF shows the diffusion of occupation probability starting from BCN. Initially, all probability is concentrated at the source. Over time, it spreads to directly connected airports, then to more distant nodes, gradually approaching the stationary distribution. Hubs accumulate probability faster, while peripheral airports remain smaller. This visualization illustrates both the spatial and temporal aspects of diffusion in the European airport network.

<a id='section64'></a>
### **6.4 First Passage Time**  


First Passage Time (FPT) measures how long it takes, for a passenger starting at a given airport (in our case BCN) to reach other airports for the first time. By simulating many random walks, we estimate the mean and success rate.

In [ ]:
def calculate_first_passage_times(
    G,
    start_node,
    target_nodes=None,
    num_simulations=5000,
    max_steps=500,
    weight_attr="flights_per_week",
):
    """
    Calculate first passage times: how long it takes to reach target nodes
    from the start node for the first time.

    Parameters:
        G: The airline network
        start_node: Starting airport
        target_nodes: List of target airports. If None, calculates for all reachable nodes
        num_simulations: Number of random walks to simulate
        max_steps: Maximum steps per walk
        weight_attr: Edge attribute for weighted walks
    """
    if target_nodes is None:
        target_nodes = [n for n in G.nodes() if n != start_node]

    # Store first passage times for each target
    passage_times = {target: [] for target in target_nodes}

    print(f"Simulating {num_simulations} walks to calculate first passage times...")

    for sim in range(num_simulations):
        # Perform a weighted random walk
        path = weighted_random_walk(
            G, start_node, steps=max_steps, weight_attr=weight_attr
        )

        # Track when each target is first visited
        visited_targets = set()
        for step, node in enumerate(path):
            if node in target_nodes and node not in visited_targets:
                passage_times[node].append(step)
                visited_targets.add(node)

    # Calculate statistics
    fpt_stats = {}
    for target, times in passage_times.items():
        if times:
            fpt_stats[target] = {
                "mean": np.mean(times),
                "median": np.median(times),
                "std": np.std(times),
                "min": np.min(times),
                "max": np.max(times),
                "success_rate": len(times) / num_simulations,
                "times": times,
            }
        else:
            fpt_stats[target] = {
                "mean": np.inf,
                "median": np.inf,
                "std": np.inf,
                "min": np.inf,
                "max": np.inf,
                "success_rate": 0.0,
                "times": [],
            }

    return fpt_stats

To make the first passage time (FPT) analysis more meaningful, we restricted the target airports to highly connected hubs. Low-traffic or remote airports can have artificially low FPTs in some simulations due to rare direct flights, which can distort the interpretation. By focusing on airports with above-median weighted degree (total weekly flights), we ensure that the FPT reflects realistic network accessibility, highlighting the speed at which passengers can reach major hubs from the starting airport BCN.

In [ ]:
# Filter highly connected airports
# Compute weighted degree (total flights per week)
weighted_degree = dict(G.degree(weight="flights_per_week"))

# Keep airports above median weighted degree
threshold = np.median(list(weighted_degree.values()))
highly_connected_airports = [
    node for node, wdeg in weighted_degree.items() if wdeg >= threshold
]

# Calculate First Passage Times (FPT)
random.seed(42)

fpt_stats = calculate_first_passage_times(
    G,
    start_node="BCN",
    target_nodes=highly_connected_airports,
    num_simulations=5000,
    max_steps=50,
    weight_attr="flights_per_week",
)

# Top 10 fastest reachable airports
top_10_fpt = sorted(
    [
        (airport, stats["mean"], stats["success_rate"])
        for airport, stats in fpt_stats.items()
        if stats["success_rate"] > 0 and airport != "BCN"
    ],
    key=lambda x: x[1],
)[:10]

fpt_data = []
for rank, (airport, mean_fpt, success_rate) in enumerate(top_10_fpt, start=1):
    airport_name = G.nodes[airport].get("name", "Unknown")
    airport_city = G.nodes[airport].get("city", "Unknown")
    fpt_data.append(
        {
            "Rank": rank,
            "Airport": airport,
            "City": airport_city,
            "Name": airport_name,
            "Mean FPT (steps)": mean_fpt,
            "Success Rate (%)": success_rate * 100,
        }
    )

df_top_10_fpt = pd.DataFrame(fpt_data)

print("\nTop 10 fastest reachable highly connected airports from BCN:")
display(df_top_10_fpt)

In [ ]:
display(df_top_10_fpt.sort_values(by="Success Rate (%)", ascending=False))

In [ ]:
# Extract airports and their mean FPTs
airports = [airport for airport, mean_fpt, success_rate in top_10_fpt]
fpt_values = [mean_fpt for airport, mean_fpt, success_rate in top_10_fpt]

plt.figure(figsize=(12, 6))
bars = plt.barh(airports, fpt_values, color="#377eb8")
plt.gca().invert_yaxis()
plt.xlabel("Mean First Passage Time (steps)")
plt.ylabel("Airport IATA code")
plt.title(
    "Top 10 Fastest Reachable Highly Connected Airports from BCN", fontweight="bold"
)
plt.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

This analysis quantifies how long it takes, on average, for a passenger starting from Barcelona (BCN) to reach other airports in the network for the first time. Each FPT value represents the mean number of steps required to reach the airport across the simulated random walks, while the success rate shows how often a passenger actually reaches the airport during the simulations.

For example, Granada (GRX) shows a mean FPT of 12.81 steps (meaning it takes, on average, about 13 random walk steps to reach Granada from Barcelona for the first time) but a very low success rate of 2.66%, meaning that although it can be reached quickly when it is visited, it is rarely reached overall due to its limited connectivity. In contrast, highly connected hubs like Madrid (MAD), Amsterdam (AMS), and Frankfurt (FRA) have slightly higher mean FPTs (~13–15 steps) but much higher success rates (30–50%), reflecting that these airports are not only reachable within a reasonable number of steps but are also frequently visited in the network.

<a id='section65'></a>
### **6.5 Failure impact analysis**  

Now we will analyze the resilience of the network by simulating the failure of critical airports and routes, measuring the impact on three key metrics: coverage, price, and duration.   

The function first computes baseline metrics by performing multiple weighted random walks from randomly selected starting airports, where edge selection is proportional to the number of flights per week. During these walks, it tracks three measures: the percentage of nodes visited (coverage), the average price per step, and the average flight duration per step. The function then simulates failures by systematically removing high-degree nodes (major hubs) or high-traffic edges (busiest routes) from the network. For each failure scenario, the same random walks are repeated on the modified network to measure the resulting coverage, average price, and duration. The impact of each failure is quantified as absolute and relative changes compared to baseline values, allowing the identification of the most critical airports or routes in terms of diffusion efficiency, travel cost, and travel time. 

In [ ]:
def analyze_failure_impact(
    G,
    weight_attr="flights_per_week",
    num_simulations=200,
    walk_steps=100,
    failure_type="node",
    print_top=10,
):
    """
    Analyze how the failure of airports or routes affects coverage, price, and duration across the network.

    Parameters:
        G: The airline network
        weight_attr: Edge attribute to use as weight (default "flights_per_week")
        num_simulations: Number of random walks per scenario
        walk_steps: Number of steps per walk
        failure_type: 'node' for airport failures, 'edge' for route failures
        print_top: print the top airports / routes
    """

    # Baseline: no failures
    baseline_coverage = []
    baseline_prices = []
    baseline_durations = []
    start_nodes = random.sample(list(G.nodes()), min(10, G.number_of_nodes()))

    for start in start_nodes:
        # Get coverage using existing function
        results = multiple_random_walks(
            G, start, num_walks=num_simulations, steps=walk_steps, walk_type="weighted"
        )
        baseline_coverage.append(results["coverage_percentage"])

        # Calculate price and duration
        total_price = 0
        total_duration = 0
        total_steps = 0

        for _ in range(num_simulations):
            current = start
            for step in range(walk_steps):
                neighbors = list(G.successors(current))
                if not neighbors:
                    break

                # Weighted random walk
                weights = [G[current][n].get(weight_attr, 1) for n in neighbors]
                next_node = random.choices(neighbors, weights=weights)[0]

                # Accumulate price and duration
                edge_data = G[current][next_node]
                total_price += edge_data.get("price", 0)
                total_duration += edge_data.get("duration", 0)
                total_steps += 1

                current = next_node

        if total_steps > 0:
            baseline_prices.append(total_price / total_steps)
            baseline_durations.append(total_duration / total_steps)

    baseline_avg_coverage = np.mean(baseline_coverage)
    baseline_avg_price = np.mean(baseline_prices)
    baseline_avg_duration = np.mean(baseline_durations)

    print(f"Baseline average coverage: {baseline_avg_coverage:.2f}%")
    print(f"Baseline average price per step: ${baseline_avg_price:.2f}")
    print(f"Baseline average duration per step: {baseline_avg_duration:.2f} minutes")

    # Identify critical nodes/edges
    if failure_type == "node":
        # High-degree nodes (major hubs)
        degree_dict = dict(G.degree())
        sorted_nodes = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)
        test_elements = [node for node, _ in sorted_nodes[:20]]
    else:
        # High-traffic edges
        edge_weights = {
            (u, v): data.get(weight_attr, 1) for u, v, data in G.edges(data=True)
        }
        sorted_edges = sorted(edge_weights.items(), key=lambda x: x[1], reverse=True)
        test_elements = [edge for edge, _ in sorted_edges[:20]]

    # Test each failure
    failure_results = []

    for element in test_elements:
        G_failed = G.copy()

        if failure_type == "node":
            node = element
            G_failed.remove_node(node)
            element_name = node
            element_info = (
                G.nodes[node].get("city", "Unknown") if node in G.nodes else "Unknown"
            )
            metric_value = degree_dict[element]
        else:
            u, v = element
            G_failed.remove_edge(u, v)
            element_name = f"{u}-{v}"
            element_info = f"{u} to {v}"
            metric_value = edge_weights[element]

        coverage_with_failure = []
        prices_with_failure = []
        durations_with_failure = []

        for start in start_nodes:
            if failure_type == "node" and start == element:
                continue
            if start not in G_failed.nodes():
                continue

            try:
                # Get coverage
                results = multiple_random_walks(
                    G_failed,
                    start,
                    num_walks=num_simulations,
                    steps=walk_steps,
                    walk_type="weighted",
                )
                coverage_with_failure.append(results["coverage_percentage"])

                # Calculate price and duration
                total_price = 0
                total_duration = 0
                total_steps = 0

                for _ in range(num_simulations):
                    current = start
                    for step in range(walk_steps):
                        neighbors = list(G_failed.successors(current))
                        if not neighbors:
                            break

                        weights = [
                            G_failed[current][n].get(weight_attr, 1) for n in neighbors
                        ]
                        next_node = random.choices(neighbors, weights=weights)[0]

                        edge_data = G_failed[current][next_node]
                        total_price += edge_data.get("price", 0)
                        total_duration += edge_data.get("duration", 0)
                        total_steps += 1

                        current = next_node

                if total_steps > 0:
                    prices_with_failure.append(total_price / total_steps)
                    durations_with_failure.append(total_duration / total_steps)
            except:
                coverage_with_failure.append(0)

        if coverage_with_failure:
            avg_coverage = np.mean(coverage_with_failure)
            coverage_drop = baseline_avg_coverage - avg_coverage
            percent_drop = (coverage_drop / baseline_avg_coverage) * 100

            avg_price = (
                np.mean(prices_with_failure)
                if prices_with_failure
                else baseline_avg_price
            )
            avg_duration = (
                np.mean(durations_with_failure)
                if durations_with_failure
                else baseline_avg_duration
            )

            price_change = avg_price - baseline_avg_price
            duration_change = avg_duration - baseline_avg_duration

            price_percent_change = (
                (price_change / baseline_avg_price) * 100
                if baseline_avg_price > 0
                else 0
            )
            duration_percent_change = (
                (duration_change / baseline_avg_duration) * 100
                if baseline_avg_duration > 0
                else 0
            )

            failure_results.append(
                {
                    "element": element_name,
                    "info": element_info,
                    "metric_value": metric_value,
                    "baseline_coverage": baseline_avg_coverage,
                    "coverage_with_failure": avg_coverage,
                    "coverage_drop": coverage_drop,
                    "percent_drop": percent_drop,
                    "baseline_price": baseline_avg_price,
                    "price_with_failure": avg_price,
                    "price_change": price_change,
                    "price_percent_change": price_percent_change,
                    "baseline_duration": baseline_avg_duration,
                    "duration_with_failure": avg_duration,
                    "duration_change": duration_change,
                    "duration_percent_change": duration_percent_change,
                    "nodes_remaining": G_failed.number_of_nodes(),
                    "edges_remaining": G_failed.number_of_edges(),
                }
            )

    failure_df = pd.DataFrame(failure_results)
    failure_df = failure_df.sort_values("coverage_drop", ascending=False).reset_index(
        drop=True
    )

    if print_top > 0:
        print("\nRESULTS:")
        display(failure_df.head(print_top))

    return failure_df

In [ ]:
# Airport failures (nodes)
random.seed(42)

node_failure_df = analyze_failure_impact(G, failure_type="node")

In [ ]:
# Route failures (edges)
random.seed(42)

edge_failure_df = analyze_failure_impact(G, failure_type="edge")

In [ ]:
def plot_failure_impact(failure_df, top_n=10, failure_type="node"):
    """
    Plot the impact of failures on coverage, price, and duration.

    Parameters:
        failure_df: Output of analyze_failure_impact()
        top_n: Number of top failures to plot
        failure_type: 'node' or 'edge'
    """
    top_failures = failure_df.head(top_n)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Coverage drop
    axes[0].barh(
        top_failures["element"], top_failures["coverage_drop"], color="#377eb8"
    )
    axes[0].invert_yaxis()
    axes[0].set_xlabel("Coverage Drop (%)")
    axes[0].set_ylabel(f"{failure_type.capitalize()} Failed")
    axes[0].set_title("Impact on Coverage", fontweight="bold")
    axes[0].grid(alpha=0.3, axis="x")

    # Price change
    axes[1].barh(top_failures["element"], top_failures["price_change"], color="#ff7f00")
    axes[1].invert_yaxis()
    axes[1].set_xlabel("Price Change ($ per step)")
    axes[1].set_title("Impact on Price", fontweight="bold")
    axes[1].grid(alpha=0.3, axis="x")

    # Duration change
    axes[2].barh(
        top_failures["element"], top_failures["duration_change"], color="#4daf4a"
    )
    axes[2].invert_yaxis()
    axes[2].set_xlabel("Duration Change (minutes per step)")
    axes[2].set_title("Impact on Duration", fontweight="bold")
    axes[2].grid(alpha=0.3, axis="x")

    plt.suptitle(
        f"Top {top_n} {failure_type.capitalize()} Failures on Network Diffusion",
        fontsize=16,
        fontweight="bold",
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
# Plot airport failures (nodes)
plot_failure_impact(node_failure_df, top_n=10, failure_type="node")

The results show that the most critical airport failure is Munich (MUC), which leads to a coverage drop of about 3%, corresponding to a ~5.8% relative reduction in network coverage. Other major hubs such as Stockholm (ARN), Frankfurt (FRA), and Barcelona (BCN) produce smaller coverage losses, typically below 1%, indicating a relatively high level of redundancy in the network structure.

From an economic perspective, airport failures also affect average price and travel duration experienced during diffusion. Some failures (e.g., FRA and MUC) slightly reduce average prices, likely due to the removal of expensive hub routes, while others (e.g., BCN, ARN, STN) lead to price increases of up to 2–3%, reflecting forced rerouting through less efficient or more costly paths. Similarly, average travel duration per step generally increases following the removal of key hubs, with the largest increases observed for MUC and FRA, suggesting longer or less direct routes after hub disruptions.

Overall, this analysis indicates that while the network is structurally robust to single-airport failures in terms of coverage, hub failures can still degrade diffusion efficiency, increasing both travel cost and travel time, which highlights the functional importance of highly connected airports beyond pure connectivity alone.

In [ ]:
# Plot route failures (edges)
plot_failure_impact(edge_failure_df, top_n=10, failure_type="edge")

The most critical route disruption is ARN–HEL (Stockholm Arlanda Airport to Helsinki Airport) which reduces network coverage by approximately 0.58% (about 1.1% relative impact). Other high-traffic routes such as BGO–OSL (Bergen Airport to Oslo Airport) and BCN–PMI (El Prat Barcelona to Palma de Mallorca Airport) show similar but slightly smaller effects, indicating that even busy routes are typically embedded in redundant regional structures that allow diffusion to reroute efficiently.

In terms of economic effects, route failures tend to cause modest increases in average price per step, typically between 0.3% and 1.6%, reflecting limited detours through slightly more expensive connections. The impact on travel duration is similarly small, with most route failures increasing average duration by less than 0.5%, suggesting that alternative paths remain relatively efficient. Interestingly, some route removals (e.g., CPH–ARN, CPH–OSL) lead to no loss or even slight improvements in coverage, highlighting cases where removing redundant or highly localized connections can marginally redirect diffusion through more efficient network regions.

Overall, these results indicate that the European airline network is highly robust to single-route failures, with limited degradation in coverage, price, or duration. While certain regional routes are important for local connectivity, system-wide diffusion dynamics are dominated by hub-level structure, making airport failures significantly more disruptive than individual route disruptions.

---
<a id='section7'></a>
## **7. Optimal routing and flight strategies**  

Finally, we implemented a route finding function that identifies flight paths between any two airports. The function can compute routes based on different criteria: the cheapest, fastest, or those with the fewest transfers. It also calculates a “best overall” route by combining multiple factors: price, duration, and the number of transfers. For this, each flight is scaled relative to the most expensive and longest flights in the network, and a weighted sum of these values is used to evaluate each route, with a small penalty for additional transfers. The route with the lowest total score is selected as the best overall.

In [ ]:
def find_routes(G, origin, destination, route_type="all"):
    """
    Find routes between origin and destination airports.

    Parameters:
        G: The airline network graph
        origin: Origin airport IATA code
        destination: Destination airport IATA code
        route_type: 'cheapest', 'fastest', 'least_transfers', 'best_overall', or 'all'
    """

    results = {}

    # Cheapest route
    if route_type in ["cheapest", "all"]:
        try:
            path_cheapest = nx.shortest_path(G, origin, destination, weight="price")
            total_price = sum(
                G[path_cheapest[i]][path_cheapest[i + 1]]["price"]
                for i in range(len(path_cheapest) - 1)
            )
            total_duration = sum(
                G[path_cheapest[i]][path_cheapest[i + 1]]["duration"]
                for i in range(len(path_cheapest) - 1)
            )
            results["cheapest"] = {
                "path": path_cheapest,
                "transfers": len(path_cheapest) - 1,
                "price": total_price,
                "duration": total_duration,
            }
        except nx.NetworkXNoPath:
            results["cheapest"] = None

    # Fastest route
    if route_type in ["fastest", "all"]:
        try:
            path_fastest = nx.shortest_path(G, origin, destination, weight="duration")
            total_price = sum(
                G[path_fastest[i]][path_fastest[i + 1]]["price"]
                for i in range(len(path_fastest) - 1)
            )
            total_duration = sum(
                G[path_fastest[i]][path_fastest[i + 1]]["duration"]
                for i in range(len(path_fastest) - 1)
            )
            results["fastest"] = {
                "path": path_fastest,
                "transfers": len(path_fastest) - 1,
                "price": total_price,
                "duration": total_duration,
            }
        except nx.NetworkXNoPath:
            results["fastest"] = None

    # Least transfers
    if route_type in ["least_transfers", "all"]:
        try:
            # Use unweighted shortest path
            path_least_transfers = nx.shortest_path(G, origin, destination)
            total_price = sum(
                G[path_least_transfers[i]][path_least_transfers[i + 1]]["price"]
                for i in range(len(path_least_transfers) - 1)
            )
            total_duration = sum(
                G[path_least_transfers[i]][path_least_transfers[i + 1]]["duration"]
                for i in range(len(path_least_transfers) - 1)
            )
            results["least_transfers"] = {
                "path": path_least_transfers,
                "transfers": len(path_least_transfers) - 1,
                "price": total_price,
                "duration": total_duration,
            }
        except nx.NetworkXNoPath:
            results["least_transfers"] = None

    # Best overall
    if route_type in ["best_overall", "all"]:
        # Normalize metrics and create composite weight
        # Get all edge attributes for normalization
        prices = [
            data["price"] for u, v, data in G.edges(data=True) if data["price"] > 0
        ]
        durations = [
            data["duration"]
            for u, v, data in G.edges(data=True)
            if data["duration"] > 0
        ]

        max_price = max(prices) if prices else 1
        max_duration = max(durations) if durations else 1

        # Create temporary graph with composite weights
        G_composite = G.copy()
        for u, v, data in G_composite.edges(data=True):
            # Normalized weighted combination
            norm_price = data["price"] / max_price
            norm_duration = data["duration"] / max_duration
            # Weight: 40% price, 40% duration, 20% hop penalty
            composite_weight = 0.4 * norm_price + 0.4 * norm_duration + 0.2
            G_composite[u][v]["composite"] = composite_weight

        try:
            path_overall = nx.shortest_path(
                G_composite, origin, destination, weight="composite"
            )
            total_price = sum(
                G[path_overall[i]][path_overall[i + 1]]["price"]
                for i in range(len(path_overall) - 1)
            )
            total_duration = sum(
                G[path_overall[i]][path_overall[i + 1]]["duration"]
                for i in range(len(path_overall) - 1)
            )
            results["best_overall"] = {
                "path": path_overall,
                "transfers": len(path_overall) - 1,
                "price": total_price,
                "duration": total_duration,
            }
        except nx.NetworkXNoPath:
            results["best_overall"] = None

    return results

In [ ]:
def obtain_iata(G, query):
    """
    Search for the IATA code.
    """
    query = query.strip().lower()
    matches = []

    for iata, data in G.nodes(data=True):
        city = data.get("city", "").lower()
        name = data.get("name", "").lower()
        country = data.get("country", "").lower()

        if query in city or query in name or query in country:
            matches.append(
                (
                    iata,
                    data.get("name", "Unknown"),
                    data.get("city", "Unknown"),
                    data.get("country", "Unknown"),
                )
            )

    if not matches:
        print("No matching airports found.")
        return

    print(f"\n\033[1m{'IATA':<6} {'Airport Name':<35} {'City':<20} Country\033[0m")
    print("-" * 80)

    for iata, name, city, country in sorted(matches):
        print(f"{iata:<6} {name:<35} {city:<20} {country}")


In [ ]:
def print_routes(G, origin, destination, route_type="all"):
    routes = find_routes(G, origin, destination, route_type)

    origin_city = G.nodes[origin].get("city", "Unknown")
    dest_city = G.nodes[destination].get("city", "Unknown")

    print(
        f"\n\033[1mROUTES FROM {origin_city} ({origin}) TO {dest_city} ({destination})\033[0m"
    )
    print("-" * 80)

    for rtype, info in routes.items():
        print(f"\n\033[1m{rtype.replace('_', ' ').title().upper()}\033[0m")

        if info is None:
            print("No route found.")
            continue

        path = info["path"]
        transfers = info["transfers"]
        price = info["price"]
        duration = info["duration"]

        # Route
        path_str = " → ".join(
            f"{a} ({G.nodes[a].get('city', 'Unknown')})" for a in path
        )
        print(f"  - Route: {path_str}")
        print(f"  - Transfers: {transfers}")
        print(f"  - Total Price: ${price:.2f}")

        duration = info["duration"]
        hours = duration // 60
        minutes = duration % 60
        print(f"  - Duration: {hours}h {minutes}m")

        print("\n  \033[4mFlight Details\033[0m")

        for i in range(len(path) - 1):
            a = path[i]
            b = path[i + 1]

            a_city = G.nodes[a].get("city", "Unknown")
            b_city = G.nodes[b].get("city", "Unknown")

            seg_price = G[a][b]["price"]

            print(f"\n    Flight {i + 1}: {a} ({a_city}) → {b} ({b_city})")
            print(f"     - Price: ${seg_price:.2f}")

            duration_ab = G[a][b]["duration"]
            hours_ab = duration_ab // 60
            minutes_ab = duration_ab % 60
            print(f"     - Duration: {hours_ab}h {minutes_ab}m")

            airlines = G[a][b].get("airlines", [])
            if airlines:
                print(f"     - Airlines: {', '.join(airlines)}")

In [ ]:
obtain_iata(G, "fiumicino")

In [ ]:
print_routes(G, "BCN", "FCO", "all")